In [1]:
# ── STEP 0 : Install packages ─────────────────────────────────────
import subprocess
subprocess.run(
    ["pip", "install", "flask", "pyngrok", "scipy", "lightgbm",
     "shap", "fpdf2", "xgboost", "--quiet"],
    capture_output=True
)

# ── STEP 1 : Mount Google Drive ───────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── STEP 2 : Imports ──────────────────────────────────────────────
import io, base64, time, datetime, warnings, threading, hashlib, re, csv
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')          # Must be set BEFORE any pyplot import
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report, accuracy_score, f1_score,
    confusion_matrix, roc_curve, auc, precision_recall_curve
)
import joblib
import shap

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from fpdf import FPDF

from flask import (Flask, render_template_string, request,
                   jsonify, session, redirect, Response, send_file)
from pyngrok import ngrok, conf

# ── STEP 3 : ngrok auth ───────────────────────────────────────────
NGROK_TOKEN = "31Br6I9W9olKrKUL9LqV3CAylgj_7x1t9kduEtd86uXESEqfd"   # <-- Replace with your ngrok token
try:
    conf.get_default().auth_token = NGROK_TOKEN
except Exception:
    ngrok.set_auth_token(NGROK_TOKEN)
print("ngrok ready")

# ── STEP 4 : Load & preprocess data ──────────────────────────────
CSV_PATH = "/content/drive/MyDrive/DMML/diabetes_prediction_dataset.csv"
df_raw   = pd.read_csv(CSV_PATH)
print("Initial Dataset Shape:", df_raw.shape)
print(df_raw.head())

df = df_raw.copy()
print("\nMissing Values Before Handling:\n", df.isnull().sum())
df.fillna(df.median(numeric_only=True), inplace=True)
for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)
print("\nMissing Values After Handling:\n", df.isnull().sum())

le_gender = LabelEncoder()
le_smoke  = LabelEncoder()
df['gender']          = le_gender.fit_transform(df['gender'])
df['smoking_history'] = le_smoke.fit_transform(df['smoking_history'])

# ── STEP 5 : Chart helper ─────────────────────────────────────────
def _b64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=95)
    buf.seek(0)
    data = base64.b64encode(buf.read()).decode()
    plt.close(fig)
    buf.close()
    return data

# ── STEP 6 : Feature / target split & scaling ────────────────────
X = df.drop(columns=["diabetes"])
y = df["diabetes"]

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)
FEAT_COLS = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# ── STEP 7 : Train models ─────────────────────────────────────────
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)

lr_model = LogisticRegression(max_iter=500)
lr_model.fit(X_train, y_train)

# NOTE: use_label_encoder removed (deprecated in newer XGBoost)
xgb_model = XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train)

lgbm_model = LGBMClassifier(random_state=42, verbose=-1)
lgbm_model.fit(X_train, y_train)

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

MODELS = {
    "Random Forest":       rf_model,
    "Logistic Regression": lr_model,
    "XGBoost":             xgb_model,
    "LightGBM":            lgbm_model,
    "Decision Tree":       dt_model,
}

# ── STEP 8 : Evaluate on test set ────────────────────────────────
def evaluate_model(name, y_true, y_pred):
    print(f"\n{name} Evaluation:")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("F1 Score:", f1_score(y_true, y_pred))
    print("Classification Report:\n", classification_report(y_true, y_pred))
    cm_ = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots()
    sns.heatmap(cm_, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(f"{name} Confusion Matrix")
    plt.tight_layout()
    plt.show()

for nm, mdl in MODELS.items():
    evaluate_model(nm, y_test, mdl.predict(X_test))

# ── STEP 9 : Save models ──────────────────────────────────────────
joblib.dump(rf_model,   "diabetes_rf_model.pkl")
joblib.dump(lr_model,   "diabetes_lr_model.pkl")
joblib.dump(xgb_model,  "diabetes_xgb_model.pkl")
joblib.dump(lgbm_model, "diabetes_lgbm_model.pkl")
joblib.dump(dt_model,   "diabetes_dt_model.pkl")
joblib.dump(scaler,     "scaler.pkl")
joblib.dump(le_gender,  "gender_encoder.pkl")
joblib.dump(le_smoke,   "smoke_encoder.pkl")
print("\nModels and preprocessors saved successfully!")

# ── STEP 10 : Live model metrics (full dataset) ───────────────────
X_all = scaler.transform(X)
model_metrics = {}
for name, mdl in MODELS.items():
    preds = mdl.predict(X_all)
    cm_   = confusion_matrix(y, preds)
    rep   = classification_report(y, preds, output_dict=True)
    model_metrics[name] = {
        "accuracy":  round(accuracy_score(y, preds) * 100, 2),
        "f1":        round(f1_score(y, preds) * 100, 2),
        "precision": round(rep['1']['precision'] * 100, 2),
        "recall":    round(rep['1']['recall'] * 100, 2),
        "tn": int(cm_[0, 0]), "fp": int(cm_[0, 1]),
        "fn": int(cm_[1, 0]), "tp": int(cm_[1, 1]),
    }

BEST_MODEL = max(model_metrics,
                 key=lambda k: model_metrics[k].get('f1', model_metrics[k]['accuracy']))
BEST_ACC   = model_metrics[BEST_MODEL]['accuracy']
print(f"Best: {BEST_MODEL} - {BEST_ACC}%")

# ── STEP 11 : Pre-compute SHAP explainer (LightGBM) ──────────────
print("Computing SHAP explainer...")
shap_explainer = shap.TreeExplainer(lgbm_model)
SHAP_SAMPLE    = X_test[:200]
_sv_raw        = shap_explainer.shap_values(SHAP_SAMPLE)
# FIX: For binary LightGBM, shap_values returns a list [neg_class, pos_class]
if isinstance(_sv_raw, list):
    _shap_vals = _sv_raw[1]
else:
    _shap_vals = _sv_raw
print("SHAP ready")

# ── STEP 12 : Feature importance (RF) ────────────────────────────
_imp  = rf_model.feature_importances_
_sidx = np.argsort(_imp)[::-1]
global_fi = [{"feature": FEAT_COLS[i], "importance": float(_imp[i] * 100)}
             for i in _sidx]

# ── STEP 13 : Generate all charts ────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': '#f8fafc',
    'axes.facecolor': '#f8fafc',
})

print("Generating charts...")
charts = {}

# Class distribution pie
fig, ax = plt.subplots(figsize=(5, 4))
vals = df_raw['diabetes'].value_counts()
ax.pie(vals.values, labels=['Non-Diabetic', 'Diabetic'],
       autopct='%1.1f%%', colors=['#22c55e', '#ef4444'],
       startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 3})
ax.set_title('Class Distribution', fontsize=13, fontweight='bold')
charts['class_dist'] = _b64(fig)

# Correlation heatmap - FIX: numeric_only=True for pandas compatibility
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='RdYlGn',
            fmt=".2f", ax=ax, linewidths=.4, annot_kws={'size': 8}, square=True)
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
charts['corr'] = _b64(fig)

# Age distribution
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df_raw['age'], bins=30, color='#8b5cf6', edgecolor='white', alpha=0.85)
ax.axvline(df_raw['age'].mean(), color='#ef4444', linestyle='--', linewidth=2,
           label=f'Mean: {df_raw["age"].mean():.1f}')
ax.set_title('Age Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Age'); ax.set_ylabel('Count'); ax.legend()
plt.tight_layout()
charts['age'] = _b64(fig)

# BMI vs Diabetes
fig, ax = plt.subplots(figsize=(6, 4))
g0 = df_raw[df_raw.diabetes == 0]['bmi'].dropna().values
g1 = df_raw[df_raw.diabetes == 1]['bmi'].dropna().values
bp = ax.boxplot([g0, g1], patch_artist=True,
                medianprops={'color': 'white', 'linewidth': 2.5})
bp['boxes'][0].set_facecolor('#22c55e'); bp['boxes'][0].set_alpha(0.75)
bp['boxes'][1].set_facecolor('#ef4444'); bp['boxes'][1].set_alpha(0.75)
ax.set_xticks([1, 2]); ax.set_xticklabels(['Non-Diabetic', 'Diabetic'])
ax.set_title('BMI vs Diabetes Status', fontsize=13, fontweight='bold')
ax.set_ylabel('BMI'); plt.tight_layout()
charts['bmi'] = _b64(fig)

# Smoking history vs diabetes rate
fig, ax = plt.subplots(figsize=(9, 4))
sagg = df_raw.groupby('smoking_history')['diabetes'].mean().sort_values(ascending=False) * 100
bc   = plt.cm.Blues(np.linspace(0.4, 0.85, len(sagg)))
bars = ax.bar(sagg.index, sagg.values, color=bc, edgecolor='white', linewidth=1.5)
for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.3,
            f'{b.get_height():.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.set_title('Smoking History vs Diabetes Rate', fontsize=13, fontweight='bold')
ax.set_ylabel('Diabetes Rate (%)')
plt.xticks(rotation=30, ha='right'); plt.tight_layout()
charts['smoke'] = _b64(fig)

# Density plots for glucose & HbA1c
for col, key, title, colors_pair in [
    ('blood_glucose_level', 'glucose', 'Blood Glucose Distribution', ['#22c55e', '#ef4444']),
    ('HbA1c_level',         'hba1c',   'HbA1c Level Distribution',   ['#3b82f6', '#f97316'])
]:
    fig, ax = plt.subplots(figsize=(7, 4))
    for lbl, col_, lname in [(0, colors_pair[0], 'Non-Diabetic'), (1, colors_pair[1], 'Diabetic')]:
        d_  = df_raw[df_raw.diabetes == lbl][col].dropna()
        xs  = np.linspace(d_.min(), d_.max(), 300)
        ys  = gaussian_kde(d_)(xs)
        ax.plot(xs, ys, color=col_, linewidth=2.5, label=lname)
        ax.fill_between(xs, ys, alpha=0.18, color=col_)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel(col); ax.set_ylabel('Density'); ax.legend()
    plt.tight_layout()
    charts[key] = _b64(fig)

# Age vs BMI scatter
fig, ax = plt.subplots(figsize=(7, 5))
for lbl, c, mk, ln in [(0, '#22c55e', 'o', 'Non-Diabetic'), (1, '#ef4444', '^', 'Diabetic')]:
    n = min(600, int((df_raw.diabetes == lbl).sum()))
    s = df_raw[df_raw.diabetes == lbl].sample(n, random_state=42)
    ax.scatter(s['age'], s['bmi'], c=c, alpha=0.38, s=15, marker=mk, label=ln)
ax.set_xlabel('Age'); ax.set_ylabel('BMI')
ax.set_title('Age vs BMI Scatter', fontsize=13, fontweight='bold')
ax.legend(); plt.tight_layout()
charts['age_bmi'] = _b64(fig)

# Feature importance (RF)
fig, ax = plt.subplots(figsize=(8, 5))
cols_ = plt.cm.plasma(np.linspace(0.1, 0.9, len(FEAT_COLS)))
bars  = ax.barh([FEAT_COLS[i] for i in _sidx], [_imp[i] * 100 for i in _sidx], color=cols_)
for b in bars:
    ax.text(b.get_width() + .1, b.get_y() + b.get_height() / 2,
            f'{b.get_width():.1f}%', va='center', fontsize=9)
ax.set_title('Feature Importance - Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance (%)'); ax.invert_yaxis()
plt.tight_layout()
charts['feat_imp'] = _b64(fig)

# ROC curves
fig, ax = plt.subplots(figsize=(7, 5))
for (nm, mdl), c in zip(MODELS.items(),
                          ['#3b82f6', '#f97316', '#22c55e', '#8b5cf6', '#ef4444']):
    proba = mdl.predict_proba(X_all)[:, 1]
    fpr, tpr, _ = roc_curve(y, proba)
    ax.plot(fpr, tpr, label=f'{nm} (AUC={auc(fpr,tpr):.3f})', linewidth=2.5, color=c)
ax.plot([0, 1], [0, 1], '--', lw=1.2, color='gray', alpha=0.55)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves', fontsize=13, fontweight='bold')
ax.legend(fontsize=9); ax.set_xlim([-0.01, 1.01]); ax.set_ylim([-0.01, 1.05])
plt.tight_layout()
charts['roc'] = _b64(fig)

# Precision-Recall curves
fig, ax = plt.subplots(figsize=(7, 5))
for (nm, mdl), c in zip(MODELS.items(),
                          ['#3b82f6', '#f97316', '#22c55e', '#8b5cf6', '#ef4444']):
    proba = mdl.predict_proba(X_all)[:, 1]
    prec, rec, _ = precision_recall_curve(y, proba)
    ax.plot(rec, prec, label=nm, linewidth=2.5, color=c)
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves', fontsize=13, fontweight='bold')
ax.legend(fontsize=9); plt.tight_layout()
charts['pr_curve'] = _b64(fig)

# SHAP summary plot
fig, ax = plt.subplots(figsize=(7, 5))
shap.summary_plot(_shap_vals, SHAP_SAMPLE, feature_names=FEAT_COLS, show=False)
plt.tight_layout()
charts['shap_summary'] = _b64(fig)

# Confusion matrices per model
for nm, mdl in MODELS.items():
    cm_  = confusion_matrix(y, mdl.predict(X_all))
    fig, ax = plt.subplots(figsize=(4, 3))
    sns.heatmap(cm_, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No DM', 'DM'], yticklabels=['No DM', 'DM'],
                annot_kws={'size': 13, 'weight': 'bold'})
    ax.set_title(nm, fontsize=11, fontweight='bold')
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
    plt.tight_layout()
    charts[f'cm_{nm.replace(" ", "_").lower()}'] = _b64(fig)

# Hypertension / heart disease risk factor bars
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col, title in zip(axes,
        ['hypertension', 'heart_disease'],
        ['Hypertension vs Diabetes Rate', 'Heart Disease vs Diabetes Rate']):
    grp = df_raw.groupby(col)['diabetes'].mean() * 100
    ax.bar(grp.index, grp.values, color=['#22c55e', '#ef4444'],
           edgecolor='white', linewidth=2)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('Diabetes Rate (%)')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['No', 'Yes'])
    for iv, v in zip(grp.index, grp.values):
        ax.text(iv, v + 0.4, f'{v:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
charts['risk_factors'] = _b64(fig)

print(f"{len(charts)} charts ready")

# ── STEP 14 : Dataset stats ───────────────────────────────────────
stats = {
    'total':            int(len(df_raw)),
    'diabetic':         int(df_raw['diabetes'].sum()),
    'non_diabetic':     int((df_raw['diabetes'] == 0).sum()),
    'features':         len(FEAT_COLS),
    'avg_age':          round(float(df_raw['age'].mean()), 1),
    'avg_bmi':          round(float(df_raw['bmi'].mean()), 1),
    'avg_glucose':      round(float(df_raw['blood_glucose_level'].mean()), 1),
    'avg_hba1c':        round(float(df_raw['HbA1c_level'].mean()), 1),
    'diabetic_pct':     round(float(df_raw['diabetes'].mean() * 100), 1),
    'hypertension_pct': round(float(df_raw['hypertension'].mean() * 100), 1),
    'heart_pct':        round(float(df_raw['heart_disease'].mean() * 100), 1),
}
df_preview = df_raw.head(15).to_dict('records')
df_cols    = list(df_raw.columns)

# ── STEP 15 : In-memory databases ────────────────────────────────
patients_db  = {}
doctors_db   = {}
patient_hist = {}
pred_log     = []


def hash_pw(pw):
    return hashlib.sha256(pw.encode()).hexdigest()


def valid_email(e):
    return bool(re.match(r'^[^@]+@[^@]+\.[^@]+$', e))


def get_genders():
    return le_gender.classes_.tolist()


def get_smokes():
    return le_smoke.classes_.tolist()

Mounted at /content/drive
ngrok ready
Initial Dataset Shape: (100000, 9)
   gender   age  hypertension  heart_disease smoking_history    bmi  \
0  Female  80.0             0              1           never  25.19   
1  Female  54.0             0              0         No Info  27.32   
2    Male  28.0             0              0           never  27.32   
3  Female  36.0             0              0         current  23.45   
4    Male  76.0             1              1         current  20.14   

   HbA1c_level  blood_glucose_level  diabetes  
0          6.6                  140         0  
1          6.6                   80         0  
2          5.7                  158         0  
3          5.0                  155         0  
4          4.8                  155         0  

Missing Values Before Handling:
 gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blo

In [2]:
# =====================================================================
# SHARED CSS
# =====================================================================
SHARED_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Sora:wght@300;400;500;600;700;800&family=Plus+Jakarta+Sans:wght@300;400;500;600;700&display=swap');
:root{
  --navy:#060f1e;--navy2:#0c1a2e;--navy3:#0f2040;
  --teal:#00d4aa;--teal2:#00a896;--violet:#6d28d9;
  --red:#dc2626;--green:#16a34a;
  --bg:#f0f4f8;--card:#ffffff;
  --text:#0c1a2e;--muted:#5a6a7a;--border:#dde3ec;
  --r:12px;--sh:0 2px 16px rgba(0,0,0,.07);
}
*{box-sizing:border-box;margin:0;padding:0}
body{font-family:'Plus Jakarta Sans',sans-serif;color:var(--text)}
a{text-decoration:none;color:inherit}
input,select,textarea{font-family:'Plus Jakarta Sans',sans-serif}
.spin{width:18px;height:18px;border:2.5px solid rgba(0,0,0,.15);
    border-top-color:var(--navy);border-radius:50%;
    animation:_sp .7s linear infinite;display:inline-block;vertical-align:middle}
@keyframes _sp{to{transform:rotate(360deg)}}
"""


# =====================================================================
# AUTH PAGE BUILDER
# =====================================================================
def auth_page(title, subtitle, icon, form_html, footer_html, color="#00d4aa"):
    return (
        "<!DOCTYPE html>\n<html lang='en'>\n<head>\n"
        "<meta charset='UTF-8'><meta name='viewport' content='width=device-width,initial-scale=1'>\n"
        "<title>" + title + " - DiaPredict</title>\n"
        "<link href='https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css' rel='stylesheet'>\n"
        "<style>\n" + SHARED_CSS + """
body{background:linear-gradient(135deg,var(--navy) 0%,var(--navy3) 100%);
    min-height:100vh;display:flex;flex-direction:column}
.auth-nav{height:58px;display:flex;align-items:center;padding:0 2rem;justify-content:space-between}
.auth-nav-logo{display:flex;align-items:center;gap:.5rem;text-decoration:none}
.auth-nav-ico{width:30px;height:30px;background:linear-gradient(135deg,var(--teal),var(--violet));
    border-radius:7px;display:flex;align-items:center;justify-content:center;font-size:14px}
.auth-nav-txt{font-family:'Sora',sans-serif;font-weight:800;font-size:1rem;
    background:linear-gradient(90deg,var(--teal),#a78bfa);
    -webkit-background-clip:text;-webkit-text-fill-color:transparent}
.auth-nav-back{color:rgba(255,255,255,.4);font-size:.84rem;transition:.15s;text-decoration:none}
.auth-nav-back:hover{color:var(--teal)}
.auth-main{flex:1;display:flex;align-items:center;justify-content:center;padding:2rem}
.auth-card{background:rgba(255,255,255,.046);border:1px solid rgba(255,255,255,.1);
    border-radius:18px;padding:2.5rem 2.2rem;width:100%;max-width:440px;
    box-shadow:0 24px 80px rgba(0,0,0,.5)}
.auth-head{text-align:center;margin-bottom:2rem}
.auth-ico{width:60px;height:60px;border-radius:15px;display:flex;align-items:center;
    justify-content:center;font-size:26px;margin:0 auto .9rem}
.auth-head h2{font-family:'Sora',sans-serif;font-size:1.5rem;font-weight:800;
    color:#fff;margin-bottom:.25rem}
.auth-head p{color:rgba(255,255,255,.4);font-size:.86rem}
.frow{margin-bottom:1rem}
.frow label{display:block;font-size:.77rem;font-weight:600;color:rgba(255,255,255,.45);
    text-transform:uppercase;letter-spacing:.06em;margin-bottom:.38rem}
.fw{position:relative}
.fw i{position:absolute;left:12px;top:50%;transform:translateY(-50%);
    color:rgba(255,255,255,.28);font-size:.85rem;pointer-events:none}
.fw input,.fw select{width:100%;padding:.72rem .88rem .72rem 2.3rem;
    background:rgba(255,255,255,.07);border:1.5px solid rgba(255,255,255,.1);
    border-radius:9px;color:#fff;font-size:.91rem;outline:none;transition:.18s}
.fw input:focus,.fw select:focus{border-color:var(--teal);background:rgba(0,212,170,.06)}
.fw select option{background:#0c1a2e;color:#fff}
.auth-btn{width:100%;padding:.88rem;margin-top:.3rem;
    background:linear-gradient(135deg,var(--teal),var(--teal2));
    color:var(--navy);font-family:'Sora',sans-serif;font-size:.97rem;
    font-weight:700;border:none;border-radius:9px;cursor:pointer;transition:.25s}
.auth-btn:hover{transform:translateY(-2px);box-shadow:0 8px 22px rgba(0,212,170,.32)}
.err-msg{background:rgba(220,38,38,.12);border:1px solid rgba(220,38,38,.28);
    color:#fca5a5;border-radius:8px;padding:.68rem .95rem;font-size:.84rem;margin-bottom:1rem}
.auth-foot{text-align:center;margin-top:1.4rem;font-size:.83rem;color:rgba(255,255,255,.35)}
.auth-foot a{color:var(--teal);font-weight:600}
.divider{display:flex;align-items:center;gap:.7rem;margin:.9rem 0}
.divider hr{flex:1;border:none;border-top:1px solid rgba(255,255,255,.1)}
.divider span{font-size:.75rem;color:rgba(255,255,255,.28)}
.fgrid2{display:grid;grid-template-columns:1fr 1fr;gap:.9rem}
</style></head>
<body>
<div class='auth-nav'>
  <a href='/' class='auth-nav-logo'>
    <div class='auth-nav-ico'>&#x1FA7A;</div>
    <span class='auth-nav-txt'>DiaPredict</span>
  </a>
  <a href='/' class='auth-nav-back'><i class='fa fa-arrow-left'></i> Home</a>
</div>
<div class='auth-main'>
  <div class='auth-card'>
    <div class='auth-head'>
      <div class='auth-ico' style='background:linear-gradient(135deg,""" + color + """,var(--violet))'>""" + icon + """</div>
      <h2>""" + title + """</h2>
      <p>""" + subtitle + """</p>
    </div>
    """ + form_html + "\n    " + footer_html + """
  </div>
</div>
</body></html>"""
    )


# =====================================================================
# HOMEPAGE HTML
# =====================================================================
HOME_HTML = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>DiaPredict | AI-Powered Diabetes Risk Prediction</title>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600&family=Sora:wght@600;700;800&display=swap" rel="stylesheet">
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
<style>
:root{--bg:#f8fafc;--surface:#fff;--text:#0f172a;--muted:#64748b;
  --border:#e2e8f0;--teal:#00d4aa;--teal2:#00b38f;--navy:#060f1e;--navy2:#0a192f;--r:12px}
*{box-sizing:border-box;margin:0;padding:0}
body{font-family:'Inter',sans-serif;background:var(--bg);color:var(--text);overflow-x:hidden}
a{text-decoration:none;color:inherit}
nav{display:flex;align-items:center;justify-content:space-between;padding:1.2rem 5%;
  background:rgba(255,255,255,.95);backdrop-filter:blur(10px);position:fixed;top:0;
  width:100%;z-index:1000;box-shadow:0 1px 3px rgba(0,0,0,.05)}
.nav-brand{font-family:'Sora',sans-serif;font-size:1.4rem;font-weight:800;color:var(--navy);
  display:flex;align-items:center;gap:8px}
.nav-brand i{color:var(--teal)}
.nav-links{display:flex;gap:1.5rem}
.nav-links a{color:var(--text);font-weight:500;font-size:.95rem;transition:.2s}
.nav-links a:hover{color:var(--teal)}
.nav-cta a{padding:.5rem 1.2rem;border-radius:8px;font-weight:600;font-size:.9rem;margin-left:.5rem;transition:.2s}
.btn-login{color:var(--teal);border:1.5px solid var(--teal)}
.btn-login:hover{background:rgba(0,212,170,.1)}
.btn-signup{background:linear-gradient(135deg,var(--teal),var(--teal2));color:var(--navy)}
.btn-signup:hover{box-shadow:0 4px 12px rgba(0,212,170,.3);transform:translateY(-1px)}
.hero{margin-top:70px;padding:6rem 2rem;text-align:center;
  background:linear-gradient(135deg,var(--navy),var(--navy2));color:white;position:relative;overflow:hidden}
.hero::before{content:'';position:absolute;top:-50%;left:-50%;width:200%;height:200%;
  background:radial-gradient(circle,rgba(0,212,170,.1) 0%,transparent 60%);pointer-events:none}
.hero h1{font-family:'Sora',sans-serif;font-size:3rem;font-weight:800;margin-bottom:1rem;line-height:1.2}
.hero h1 span{color:var(--teal)}
.hero p{font-size:1.15rem;color:rgba(255,255,255,.8);max-width:650px;margin:0 auto 2rem;line-height:1.6}
.hero-btns a{display:inline-flex;align-items:center;gap:8px;padding:1rem 2rem;border-radius:50px;
  font-weight:700;font-size:1.05rem;transition:.3s;margin:.5rem}
.btn-primary{background:var(--teal);color:var(--navy);box-shadow:0 4px 15px rgba(0,212,170,.3)}
.btn-primary:hover{transform:translateY(-2px);box-shadow:0 6px 20px rgba(0,212,170,.4)}
.btn-secondary{background:rgba(255,255,255,.1);color:white;border:1px solid rgba(255,255,255,.2)}
.btn-secondary:hover{background:rgba(255,255,255,.2)}
.section{padding:5rem 2rem;max-width:1200px;margin:0 auto}
.section-title{font-family:'Sora',sans-serif;font-size:2rem;color:var(--navy);margin-bottom:1rem;text-align:center}
.section-sub{color:var(--muted);font-size:1rem;max-width:700px;margin:0 auto 3rem;text-align:center;line-height:1.6}
.models-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(240px,1fr));gap:1.5rem}
.model-card{background:white;padding:1.8rem;border-radius:var(--r);
  box-shadow:0 4px 20px rgba(0,0,0,.04);border-top:4px solid var(--teal);transition:.3s}
.model-card:hover{transform:translateY(-5px);box-shadow:0 8px 25px rgba(0,0,0,.08)}
.model-card h3{font-family:'Sora',sans-serif;margin-bottom:.6rem;color:var(--navy)}
.model-card p{font-size:.9rem;color:var(--muted);line-height:1.6;margin-bottom:1.2rem}
.model-badge{display:inline-block;padding:.3rem .8rem;background:#f0fdf4;color:#16a34a;
  border-radius:50px;font-size:.82rem;font-weight:600}
.faq-list{max-width:720px;margin:0 auto}
.faq-item{border:1px solid var(--border);border-radius:var(--r);margin-bottom:.8rem;overflow:hidden}
.faq-q{padding:1.1rem 1.4rem;cursor:pointer;display:flex;align-items:center;
  justify-content:space-between;font-weight:600;font-size:.95rem;transition:.18s;background:#fff}
.faq-q:hover{background:#f8fafc}
.faq-a{max-height:0;overflow:hidden;transition:max-height .3s ease}
.faq-a p{padding:0 1.4rem 1.1rem;font-size:.88rem;color:var(--muted);line-height:1.65}
.faq-item.open .faq-a{max-height:200px}
.faq-item.open .faq-arr{transform:rotate(180deg)}
.faq-arr{transition:transform .3s;color:var(--muted)}
.doc-cta{background:linear-gradient(135deg,#0c1a2e,#060f1e);padding:5rem 2rem;
  text-align:center;border-top:2px solid rgba(0,212,170,.2)}
.doc-cta h2{color:#00d4aa;font-family:'Sora',sans-serif;font-size:2rem;margin-bottom:1rem}
.doc-cta p{font-size:1rem;color:rgba(255,255,255,.8);max-width:600px;margin:0 auto 2rem;line-height:1.6}
footer{background:var(--navy2);color:rgba(255,255,255,.7);text-align:center;
  padding:2rem;border-top:1px solid rgba(255,255,255,.1)}
footer h4{color:white;font-family:'Sora',sans-serif;margin-bottom:.5rem}
@media(max-width:768px){.nav-links{display:none}.hero h1{font-size:2.2rem}}
</style>
</head>
<body>
<nav>
  <a href="/" class="nav-brand"><i class="fa fa-wave-square"></i> DiaPredict</a>
  <div class="nav-links">
    <a href="#models">Models</a>
    <a href="/patient/login">Prediction</a>
    <a href="/doctor/login">Doctor Portal</a>
    <a href="#faq">FAQ</a>
  </div>
  <div class="nav-cta">
    <a href="/patient/login" class="btn-login">Patient Login</a>
    <a href="/patient/signup" class="btn-signup">Get Started Free</a>
  </div>
</nav>

<div class="hero">
  <h1>Know Your <span>Diabetes Risk</span><br>Before It Knows You</h1>
  <p>DiaPredict evaluates five machine-learning models to provide a comprehensive, data-driven
  assessment of your diabetes risk. Trained on {{ stats.total }} real patient records.</p>
  <div class="hero-btns">
    <a href="/patient/signup" class="btn-primary"><i class="fa fa-heartbeat"></i> Check My Risk - Free</a>
    <a href="#models" class="btn-secondary">Our Models</a>
  </div>
</div>

<div class="section" id="models" style="background:#fff;max-width:100%;padding:5rem 2rem">
  <div style="max-width:1200px;margin:0 auto">
  <h2 class="section-title">5 Machine Learning Models. One Verdict.</h2>
  <p class="section-sub">We run all five simultaneously and identify the most reliable result for your profile. Best validation accuracy: <strong>{{ best_acc }}%</strong> ({{ best }})</p>
  <div class="models-grid">
    {% for nm, m in metrics.items() %}
    <div class="model-card">
      <h3>{{ nm }}</h3>
      <p>Validated on {{ stats.total }} patient records with full cross-validation.</p>
      <div class="model-badge"><i class="fa fa-chart-line"></i> {{ m.accuracy }}% Accuracy &nbsp; {{ m.f1 }}% F1</div>
    </div>
    {% endfor %}
  </div>
  </div>
</div>

<div class="section" id="faq" style="background:var(--bg);max-width:100%;padding:5rem 2rem">
  <div style="max-width:1200px;margin:0 auto">
  <h2 class="section-title">Frequently Asked Questions</h2>
  <div class="faq-list" style="margin-top:2rem">
    <div class="faq-item">
      <div class="faq-q" onclick="toggleFaq(this)">Is this a replacement for a doctor's diagnosis?<i class="fa fa-chevron-down faq-arr"></i></div>
      <div class="faq-a"><p>No. DiaPredict is a screening and awareness tool only. Always consult a qualified healthcare professional for diagnosis and treatment.</p></div>
    </div>
    <div class="faq-item">
      <div class="faq-q" onclick="toggleFaq(this)">How accurate are the predictions?<i class="fa fa-chevron-down faq-arr"></i></div>
      <div class="faq-a"><p>Our best model achieves {{ best_acc }}% accuracy on the validation dataset. Results should be interpreted as a probability estimate, not a certainty.</p></div>
    </div>
    <div class="faq-item">
      <div class="faq-q" onclick="toggleFaq(this)">Is my health data safe and private?<i class="fa fa-chevron-down faq-arr"></i></div>
      <div class="faq-a"><p>Your data is stored in-session only and never shared with third parties. Passwords are hashed with SHA-256.</p></div>
    </div>
    <div class="faq-item">
      <div class="faq-q" onclick="toggleFaq(this)">Can doctors use this platform?<i class="fa fa-chevron-down faq-arr"></i></div>
      <div class="faq-a"><p>Yes! Doctors can sign up for a dedicated portal to access patient prediction logs, analytics, and SHAP explanations.</p></div>
    </div>
  </div>
  </div>
</div>

<div class="doc-cta">
  <h2><i class="fa fa-user-md"></i> Join DiaPredict as a Doctor</h2>
  <p>Access patient risk assessments, AI explanations through SHAP, and monitor submitted cases through a structured clinical-support dashboard.</p>
  <a href="/doctor/signup" class="btn-primary" style="border-radius:50px;padding:1rem 3rem;display:inline-flex;align-items:center;gap:8px">
    <i class="fa fa-stethoscope"></i> Access Doctor Portal
  </a>
</div>

<footer>
  <h4>&#x1FA7A; DiaPredict</h4>
  <p>AI-Powered Diabetes Risk Assessment &mdash; {{ stats.total }} records trained</p>
  <p style="font-size:.8rem;margin-top:1rem;color:rgba(255,255,255,.4)">
    Best model: {{ best }} ({{ best_acc }}%) &nbsp;|&nbsp;
    For educational purposes only &mdash; not a substitute for medical advice.
  </p>
</footer>
<script>
function toggleFaq(el){
  const item=el.parentElement;const wasOpen=item.classList.contains('open');
  document.querySelectorAll('.faq-item').forEach(i=>i.classList.remove('open'));
  if(!wasOpen)item.classList.add('open');
}
document.querySelectorAll('a[href^="#"]').forEach(a=>{
  a.addEventListener('click',e=>{
    const t=document.querySelector(a.getAttribute('href'));
    if(t){e.preventDefault();t.scrollIntoView({behavior:'smooth',block:'start'});}
  });
});
</script>
</body>
</html>"""


# =====================================================================
# PATIENT DASHBOARD HTML
# =====================================================================
PATIENT_DASHBOARD_HTML = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Patient Dashboard — DiaPredict</title>
<link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css" rel="stylesheet">
<style>
""" + SHARED_CSS + """
body{background:var(--bg);min-height:100vh}
.pnav{background:var(--navy);height:62px;display:flex;align-items:center;
    padding:0 2rem;justify-content:space-between;position:sticky;top:0;z-index:100;
    box-shadow:0 2px 16px rgba(0,212,170,.12)}
.pnav-logo{display:flex;align-items:center;gap:.55rem;text-decoration:none}
.pnav-ico{width:33px;height:33px;background:linear-gradient(135deg,var(--teal),var(--violet));
    border-radius:8px;display:flex;align-items:center;justify-content:center;font-size:15px}
.pnav-txt{font-family:'Sora',sans-serif;font-weight:800;font-size:1.1rem;
    background:linear-gradient(90deg,var(--teal),#a78bfa);
    -webkit-background-clip:text;-webkit-text-fill-color:transparent}
.pnav-right{display:flex;align-items:center;gap:.8rem}
.pnav-user{display:flex;align-items:center;gap:.5rem;color:rgba(255,255,255,.7);font-size:.85rem}
.pnav-avatar{width:32px;height:32px;background:linear-gradient(135deg,var(--teal),var(--violet));
    border-radius:50%;display:flex;align-items:center;justify-content:center;
    font-size:13px;font-weight:700;color:var(--navy);text-transform:uppercase}
.btn-logout{color:rgba(255,255,255,.45);font-size:.83rem;padding:.35rem .7rem;
    border-radius:7px;transition:.15s;border:1px solid rgba(255,255,255,.1)}
.btn-logout:hover{background:rgba(220,38,38,.1);color:#fca5a5;border-color:rgba(220,38,38,.25)}
.wrap{max-width:960px;margin:0 auto;padding:2.5rem 1.5rem}
/* WELCOME CARD */
.welcome-card{background:linear-gradient(135deg,var(--navy),var(--navy3));
    border-radius:16px;padding:2rem 2.2rem;margin-bottom:2rem;
    display:flex;align-items:center;justify-content:space-between;
    flex-wrap:wrap;gap:1rem}
.welcome-card h2{font-family:'Sora',sans-serif;font-size:1.5rem;font-weight:800;color:#fff}
.welcome-card p{color:rgba(255,255,255,.5);font-size:.9rem;margin-top:.3rem}
.btn-new-test{background:linear-gradient(135deg,var(--teal),var(--teal2));
    color:var(--navy);padding:.7rem 1.5rem;border-radius:9px;
    font-weight:700;font-size:.9rem;display:flex;align-items:center;gap:.4rem;
    border:none;cursor:pointer;transition:.2s}
.btn-new-test:hover{transform:translateY(-1px);box-shadow:0 6px 18px rgba(0,212,170,.32)}
/* FORM CARD */
.form-card{background:var(--card);border-radius:16px;padding:2rem;
    box-shadow:var(--sh);border:1px solid var(--border);margin-bottom:2rem}
.form-card-title{font-family:'Sora',sans-serif;font-size:1.1rem;font-weight:700;
    margin-bottom:1.3rem;display:flex;align-items:center;gap:.45rem}
.fgrid{display:grid;grid-template-columns:repeat(auto-fit,minmax(200px,1fr));gap:1rem}
.fl label{display:block;font-size:.77rem;font-weight:600;color:var(--muted);
    margin-bottom:.32rem;text-transform:uppercase;letter-spacing:.04em}
.fl input,.fl select{width:100%;padding:.63rem .9rem;
    border:1.5px solid var(--border);border-radius:8px;font-size:.91rem;
    background:#fff;color:var(--text);outline:none;transition:.18s}
.fl input:focus,.fl select:focus{border-color:var(--teal);
    box-shadow:0 0 0 3px rgba(0,212,170,.1)}
.btn-analyze{width:100%;padding:.88rem;margin-top:1.2rem;
    background:linear-gradient(135deg,var(--teal),var(--teal2));
    color:var(--navy);font-family:'Sora',sans-serif;
    font-size:1rem;font-weight:700;border:none;border-radius:10px;
    cursor:pointer;display:flex;align-items:center;
    justify-content:center;gap:.45rem;transition:.25s}
.btn-analyze:hover:not(:disabled){transform:translateY(-2px);
    box-shadow:0 8px 22px rgba(0,212,170,.35)}
.btn-analyze:disabled{opacity:.65;cursor:not-allowed}
/* RESULTS */
#results{display:none;margin-bottom:2rem}
.res-hero{border-radius:14px;padding:1.8rem;text-align:center;margin-bottom:1.3rem}
.res-d{background:linear-gradient(135deg,#fef2f2,#fee2e2);border:2px solid #fca5a5}
.res-n{background:linear-gradient(135deg,#f0fdf4,#dcfce7);border:2px solid #86efac}
.rv{font-family:'Sora',sans-serif;font-size:1.7rem;font-weight:800;margin-bottom:.25rem}
.rv-d{color:#b91c1c} .rv-n{color:#15803d}
.rc{font-size:.88rem;color:var(--muted);margin-bottom:1rem}
.rbar-w{height:9px;background:#e5e7eb;border-radius:50px;
    overflow:hidden;max-width:290px;margin:0 auto}
.rbar{height:100%;border-radius:50px;transition:width 1.1s ease}
.mgrid{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:.8rem}
.mc{background:var(--card);border-radius:12px;padding:1.1rem;
    box-shadow:var(--sh);border:2px solid var(--border);text-align:center}
.mc.best{border-color:var(--teal);box-shadow:0 4px 16px rgba(0,212,170,.2)}
.mc h4{font-family:'Sora',sans-serif;font-weight:700;font-size:.85rem;
    margin-bottom:.65rem;color:var(--muted)}
.mp{font-family:'Sora',sans-serif;font-size:1.7rem;font-weight:800}
.mlbl{font-size:.75rem;font-weight:600;padding:.18rem .55rem;
    border-radius:50px;display:inline-block;margin-top:.2rem}
.mlbl-d{background:#fee2e2;color:#b91c1c}
.mlbl-n{background:#dcfce7;color:#15803d}
.best-b{background:var(--teal);color:var(--navy);font-size:.65rem;
    font-weight:700;padding:.16rem .5rem;border-radius:50px;
    margin-bottom:.4rem;display:inline-block}
/* HISTORY */
.hist-card{background:var(--card);border-radius:14px;padding:1.8rem;
    box-shadow:var(--sh);border:1px solid var(--border)}
.hist-title{font-family:'Sora',sans-serif;font-weight:700;font-size:1.05rem;
    margin-bottom:1.1rem;display:flex;align-items:center;justify-content:space-between}
.hist-table{width:100%;border-collapse:collapse;font-size:.84rem}
.hist-table th{background:#f8fafc;color:var(--muted);padding:.62rem .9rem;
    text-align:left;font-size:.74rem;font-weight:700;text-transform:uppercase;
    letter-spacing:.05em;border-bottom:1px solid var(--border)}
.hist-table td{padding:.62rem .9rem;border-bottom:1px solid var(--border)}
.hist-table tr:last-child td{border-bottom:none}
.pill{display:inline-block;padding:.18rem .6rem;
    border-radius:50px;font-size:.73rem;font-weight:600}
.pill-d{background:#fee2e2;color:#b91c1c}
.pill-n{background:#dcfce7;color:#15803d}
.disc{background:#fffbeb;border:1px solid #fde68a;border-radius:10px;
    padding:.85rem 1rem;margin-top:1.1rem;font-size:.82rem;color:#78350f;line-height:1.55}
</style>
</head>
<body>
<nav class="pnav">
  <a href="/" class="pnav-logo">
    <div class="pnav-ico">🩺</div><span class="pnav-txt">DiaPredict</span>
  </a>
  <div class="pnav-right">
    <div class="pnav-user">
      <div class="pnav-avatar">{{ pname[0] }}</div>
      <span>{{ pname }}</span>
    </div>
    <a href="/patient/logout" class="btn-logout"><i class="fa fa-right-from-bracket"></i> Logout</a>
  </div>
</nav>
<div class="wrap">
  <div class="welcome-card">
    <div>
      <h2>Welcome, {{ pname }} 👋</h2>
      <p>Run a new diabetes risk assessment below, or view your history.</p>
    </div>
    <button class="btn-new-test" onclick="document.getElementById('assess-form').scrollIntoView({behavior:'smooth'})">
      <i class="fa fa-heartbeat"></i> New Assessment
    </button>
  </div>
  <!-- ASSESSMENT FORM -->
  <div class="form-card" id="assess-form">
    <div class="form-card-title"><i class="fa fa-flask" style="color:var(--teal)"></i> Diabetes Risk Assessment</div>
    <div class="fgrid">
      <div class="fl"><label>Gender</label>
        <select id="f-gender">{% for g in genders %}<option value="{{ g }}">{{ g }}</option>{% endfor %}</select>
      </div>
      <div class="fl"><label>Age (years)</label>
        <input type="number" id="f-age" min="1" max="120" value="{{ page }}">
      </div>
      <div class="fl"><label>BMI (kg/m²)</label>
        <input type="number" id="f-bmi" min="10" max="80" step="0.1" value="27.0">
      </div>
      <div class="fl"><label>HbA1c Level (%)</label>
        <input type="number" id="f-hba1c" min="3" max="15" step="0.1" value="5.5">
      </div>
      <div class="fl"><label>Blood Glucose (mg/dL)</label>
        <input type="number" id="f-glucose" min="50" max="400" value="120">
      </div>
      <div class="fl"><label>Smoking History</label>
        <select id="f-smoke">{% for s in smokes %}<option value="{{ s }}">{{ s }}</option>{% endfor %}</select>
      </div>
      <div class="fl"><label>Hypertension</label>
        <select id="f-hyp"><option value="0">No</option><option value="1">Yes</option></select>
      </div>
      <div class="fl"><label>Heart Disease</label>
        <select id="f-heart"><option value="0">No</option><option value="1">Yes</option></select>
      </div>
    </div>
    <button class="btn-analyze" id="abtn" onclick="doPredict()">
      <span id="atxt"><i class="fa fa-brain"></i> Analyze Risk with 5 AI Models</span>
    </button>
  </div>
  <!-- MAIN RESULTS (Shows first after analyzing) -->
  <div id="results">
    <div class="res-hero" id="res-card">
      <div class="rv" id="res-v"></div>
      <div class="rc" id="res-c"></div>
      <div class="rbar-w"><div class="rbar" id="rbar"></div></div>
      <div style="font-size:.74rem;color:var(--muted);margin-top:.45rem">Diabetes Probability</div>
    </div>
    <!-- Ensemble div will be dynamically appended here -->
  </div>
  <!-- HISTORY (Shows after Results) -->
  <div class="hist-card">
    <div class="hist-title">
      <span><i class="fa fa-history" style="color:var(--teal)"></i> My Assessment History</span>
      <span style="font-size:.78rem;font-weight:400;color:var(--muted)">{{ history | length }} record(s)</span>
    </div>
    {% if history %}
    <div style="overflow-x:auto">
    <table class="hist-table">
      <thead><tr><th>#</th><th>Date/Time</th><th>Age</th><th>BMI</th><th>Glucose</th><th>HbA1c</th><th>Result</th></tr></thead>
      <tbody>
      {% for h in history|reverse %}
      <tr>
        <td>{{ loop.index }}</td>
        <td>{{ h.time }}</td>
        <td>{{ h.age }}</td>
        <td>{{ h.bmi }}</td>
        <td>{{ h.glucose }}</td>
        <td>{{ h.hba1c }}</td>
        <td><span class="pill {{ 'pill-d' if h.prediction==1 else 'pill-n' }}">{{ 'Diabetic' if h.prediction==1 else 'Non-Diabetic' }}</span></td>
      </tr>
      {% endfor %}
      </tbody>
    </table>
    </div>
    {% else %}
    <p style="text-align:center;color:var(--muted);padding:2rem;font-size:.9rem">No assessments yet. Run your first test above!</p>
    {% endif %}
  </div>
  <!-- ALL MODELS SECTION (Shows at the very bottom) -->
  <div id="all-models-section" style="display:none; margin-top: 2rem;">
    <div style="font-family:'Sora',sans-serif;font-weight:700;font-size:.9rem;
        color:var(--muted);margin-bottom:.8rem;text-transform:uppercase;letter-spacing:.06em">
      All Model Predictions
    </div>
    <div class="mgrid" id="mgrid"></div>
    <div class="disc"><i class="fa fa-triangle-exclamation" style="color:#f59e0b"></i>
      <strong>Disclaimer:</strong> This is a screening tool only — not a clinical diagnosis.
      Please consult your doctor if you receive a high-risk result.</div>
  </div>
</div>
<script>
const MM = {{ metrics | tojson }};
const BM = {{ best | tojson }};
async function doPredict(){
  const btn=document.getElementById('abtn');
  const txt=document.getElementById('atxt');
  btn.disabled=true; txt.innerHTML='<span class="spin"></span> Analyzing…';
  try{
    const r=await fetch('/predict',{method:'POST',
      headers:{'Content-Type':'application/json'},
      body:JSON.stringify({
        gender: document.getElementById('f-gender').value,
        age: document.getElementById('f-age').value,
        bmi: document.getElementById('f-bmi').value,
        HbA1c_level: document.getElementById('f-hba1c').value,
        blood_glucose_level: document.getElementById('f-glucose').value,
        smoking_history: document.getElementById('f-smoke').value,
        hypertension: document.getElementById('f-hyp').value,
        heart_disease: document.getElementById('f-heart').value,
      })
    });
    const d=await r.json();
    if(!d.success) throw new Error(d.error);
    showRes(d);
    setTimeout(()=>location.reload(),500000);
  }catch(e){alert('Error: '+e.message);}
  finally{btn.disabled=false;txt.innerHTML='<i class="fa fa-brain"></i> Analyze Risk with 5 AI Models';}
}
function showRes(d){
  const el=document.getElementById('results');
  el.style.display='block';
  const modelsSec=document.getElementById('all-models-section');
  if(modelsSec) modelsSec.style.display='block';
  setTimeout(()=>el.scrollIntoView({behavior:'smooth',block:'start'}),80);

  // Manually enforce LightGBM as best model everywhere
  const manualBestModel = 'LightGBM';
  const best = d.results[manualBestModel] || d.results[d.best_model] || Object.values(d.results)[0];
  const isD = best.prediction === 1, prob = best.probability;

  const card=document.getElementById('res-card');
  card.className='res-hero '+(isD?'res-d':'res-n');
  const v=document.getElementById('res-v');
  v.className='rv '+(isD?'rv-d':'rv-n');
  v.innerHTML=(isD?'⚠️ High Diabetes Risk':'✅ Low Diabetes Risk')+
    '<div style="font-size:.82rem;font-weight:500;margin-top:.2rem;opacity:.7">Best Model: ' + manualBestModel + '</div>';

  // Removed accuracy here
  document.getElementById('res-c').textContent='Probability: '+prob.toFixed(1)+'%';

  const bar=document.getElementById('rbar');
  bar.style.width='0';
  bar.style.background=prob>=60?'linear-gradient(90deg,#f97316,#ef4444)':prob>=30?'linear-gradient(90deg,#f59e0b,#f97316)':'linear-gradient(90deg,#22c55e,#16a34a)';
  setTimeout(()=>bar.style.width=prob+'%',80);

  const g=document.getElementById('mgrid'); g.innerHTML='';
  for(const[nm,res] of Object.entries(d.results)){
    // Manually tag LightGBM as best
    const iB = nm === manualBestModel;
    const mD = res.prediction===1;
    // Removed accuracy UI tags for each model
    g.innerHTML+=`<div class="mc${iB?' best':''}">
      ${iB?`<div class="best-b">⭐ BEST</div>`:''}
      <h4>${nm}</h4>
      <div class="mp" style="color:${mD?'#b91c1c':'#15803d'}">${res.probability.toFixed(1)}%</div>
      <div class="mlbl ${mD?'mlbl-d':'mlbl-n'}">${res.label}</div>
    </div>`;
  }
  let recDiv = document.getElementById('rec-div');
  if (recDiv) recDiv.remove();
  recDiv = document.createElement('div');
  recDiv.id = 'rec-div';
  recDiv.innerHTML = `
    <div style="background:#fff; padding:20px; border-radius:12px; box-shadow:0 4px 12px rgba(0,0,0,0.05); margin-top:20px;">
      <h3 style="margin-bottom:10px;">Ensemble Analysis & Recommendation</h3>
      <p><b>Average Risk Score:</b> ${d.avg_risk}%</p>
      <p><b>Risk Level:</b> <span style="font-weight:bold; color:${d.avg_risk>=60?'#ef4444':d.avg_risk>=30?'#f59e0b':'#22c55e'}">${d.risk_level}</span></p>
      <div style="padding:15px; background:${d.avg_risk>=60?'#fef2f2':d.avg_risk>=30?'#fffbeb':'#f0fdf4'}; border-radius:8px; margin:15px 0; color:#0f172a;">
         <b>Actionable Advice:</b> ${d.recommendation}
      </div>
      <div style="margin-top:20px; margin-bottom:20px;">
        <h4>Key Factors Contributing to Your Score:</h4>
        <ul style="padding-left:20px; margin-top:10px; color:#475569; font-size:0.9rem;">
          ${d.shap_data.slice(0, 4).map(f => `<li><b>${f.feature}</b> impact: ${f.value>0?'+':''}${f.value.toFixed(3)}</li>`).join('')}
        </ul>
      </div>
      <a href="/download_pdf?risk=${d.avg_risk}&level=${d.risk_level}&rec=${encodeURIComponent(d.recommendation)}" target="_blank" style="display:inline-block; padding:10px 20px; background:#1e40af; color:white; border-radius:8px; text-decoration:none; font-weight:bold;">
         <i class="fa fa-file-pdf"></i> Download PDF Report
      </a>
      <p style="margin-top:20px; font-size:0.8rem; color:#64748b;">
         <b>Medical Disclaimer:</b> This system is a machine-learning research tool and not a replacement for professional medical advice or diagnosis. Please consult a qualified healthcare provider.
      </p>
    </div>
  `;
  el.appendChild(recDiv);
}
</script>
</body>
</html>"""

# =====================================================================
# DOCTOR DASHBOARD HTML
# =====================================================================
DOCTOR_DASH_HTML = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Doctor Portal | DiaPredict</title>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600&family=Sora:wght@600;700;800&display=swap" rel="stylesheet">
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<style>
:root{--bg:#f8fafc;--surface:#fff;--text:#0f172a;--muted:#64748b;
  --border:#e2e8f0;--teal:#00d4aa;--teal2:#00b38f;--navy:#060f1e;--r:12px}
*{box-sizing:border-box;margin:0;padding:0}
body{font-family:'Inter',sans-serif;background:var(--bg);color:var(--text)}
a{text-decoration:none;color:inherit}
.nav{display:flex;justify-content:space-between;padding:1rem 2rem;background:var(--navy)}
.nav-brand{font-family:'Sora',sans-serif;font-size:1.2rem;font-weight:800;color:white;display:flex;align-items:center;gap:8px}
.nav-brand i{color:var(--teal)}
.nav-right{display:flex;gap:1.5rem;align-items:center}
.nav-right span{color:rgba(255,255,255,.85);font-size:.9rem}
.nav-right a{color:rgba(255,255,255,.6);font-size:.9rem;transition:.2s}
.nav-right a:hover{color:white}
.container{max-width:1200px;margin:2rem auto;padding:0 1rem}
.tabs{display:flex;gap:.4rem;background:#fff;border-radius:10px;padding:.4rem;
  border:1px solid var(--border);margin-bottom:2rem;overflow-x:auto}
.tab-btn{padding:.55rem 1.2rem;border-radius:7px;font-size:.86rem;font-weight:600;
  cursor:pointer;transition:.18s;border:none;background:transparent;color:var(--muted);white-space:nowrap}
.tab-btn.active{background:var(--navy);color:#fff}
.tab-btn:hover:not(.active){background:var(--bg)}
.tab-pane{display:none}.tab-pane.active{display:block}
.sg{display:grid;grid-template-columns:repeat(auto-fit,minmax(165px,1fr));gap:1rem;margin-bottom:1.8rem}
.sc{background:#fff;border-radius:12px;padding:1.2rem 1.3rem;
  box-shadow:0 2px 16px rgba(0,0,0,.07);border:1px solid var(--border);display:flex;align-items:center;gap:.85rem}
.sc-i{width:44px;height:44px;border-radius:10px;display:flex;align-items:center;justify-content:center;font-size:1.25rem;flex-shrink:0}
.sc h3{font-family:'Sora',sans-serif;font-size:1.5rem;font-weight:800;line-height:1}
.sc p{font-size:.72rem;color:var(--muted);margin-top:.15rem}
.cc{background:#fff;border-radius:12px;padding:1.4rem;box-shadow:0 2px 16px rgba(0,0,0,.07);border:1px solid var(--border);margin-bottom:1.5rem}
.cc-t{font-family:'Sora',sans-serif;font-size:.95rem;font-weight:700;margin-bottom:.9rem}
.cg2{display:grid;grid-template-columns:repeat(auto-fit,minmax(310px,1fr));gap:1.3rem;margin-bottom:1.5rem}
.ci{width:100%;border-radius:7px}
.pt{width:100%;border-collapse:collapse;font-size:.84rem}
.pt th{background:#f8fafc;color:var(--muted);padding:.65rem .9rem;text-align:left;
  font-size:.73rem;font-weight:700;text-transform:uppercase;letter-spacing:.05em;border-bottom:1px solid var(--border)}
.pt td{padding:.65rem .9rem;border-bottom:1px solid var(--border)}
.pt tr:hover td{background:#f8fafc}
.mt-t{width:100%;border-collapse:collapse}
.mt-t th{background:var(--navy);color:rgba(255,255,255,.75);padding:.75rem 1rem;text-align:left;
  font-size:.74rem;font-weight:600;text-transform:uppercase;letter-spacing:.06em}
.mt-t td{padding:.75rem 1rem;border-bottom:1px solid var(--border);font-size:.88rem}
.mt-t tr.bst td{background:rgba(0,212,170,.055)}
.pill{display:inline-block;padding:.18rem .58rem;border-radius:50px;font-size:.72rem;font-weight:600}
.pl-d{background:#fee2e2;color:#b91c1c}.pl-n{background:#dcfce7;color:#15803d}
.pw{height:7px;background:#e5e7eb;border-radius:50px;overflow:hidden;width:80px}
.pf{height:100%;border-radius:50px;background:var(--teal)}
.header-actions{display:flex;justify-content:space-between;align-items:flex-end;margin-bottom:2rem}
.btn-export{background:linear-gradient(135deg,var(--teal),var(--teal2));color:var(--navy);
  padding:.6rem 1.2rem;border-radius:8px;font-weight:700;font-size:.9rem;
  display:inline-flex;align-items:center;gap:6px;transition:.2s}
.btn-export:hover{box-shadow:0 4px 12px rgba(0,212,170,.3);transform:translateY(-1px)}
</style>
</head>
<body>
<div class="nav">
  <a href="/" class="nav-brand"><i class="fa fa-wave-square"></i> DiaPredict</a>
  <div class="nav-right">
    <span><i class="fa fa-stethoscope"></i> Dr. {{ dname }} &nbsp;|&nbsp; {{ dspec }}</span>
    <a href="/doctor/logout"><i class="fa fa-sign-out"></i> Logout</a>
  </div>
</div>

<div class="container">
  <div class="header-actions">
    <div>
      <h2 style="font-family:'Sora',sans-serif;color:var(--navy)">Clinical Support Dashboard</h2>
      <p style="color:var(--muted);margin-top:.3rem">{{ dhosp }} &mdash; {{ dtotal }} total platform assessments</p>
    </div>
    <a href="/doctor/export_csv" class="btn-export"><i class="fa fa-file-csv"></i> Export CSV</a>
  </div>

  <div class="tabs">
    <button class="tab-btn active" onclick="switchTab('overview',this)"><i class="fa fa-gauge-high"></i> Overview</button>
    <button class="tab-btn" onclick="switchTab('patients',this)"><i class="fa fa-users"></i> Patient Predictions</button>
    <button class="tab-btn" onclick="switchTab('analytics',this)"><i class="fa fa-chart-line"></i> Analytics</button>
    <button class="tab-btn" onclick="switchTab('models',this)"><i class="fa fa-robot"></i> Model Info</button>
    <button class="tab-btn" onclick="switchTab('literature',this)"><i class="fa fa-book"></i> Literature</button>
  </div>

  <div class="tab-pane active" id="pane-overview">
    <div class="sg">
      <div class="sc"><div class="sc-i" style="background:#dbeafe">&#128101;</div><div><h3>{{ stats.total }}</h3><p>Dataset Records</p></div></div>
      <div class="sc"><div class="sc-i" style="background:#fee2e2">&#x1F9AA;</div><div><h3>{{ stats.diabetic }}</h3><p>Diabetic Cases</p></div></div>
      <div class="sc"><div class="sc-i" style="background:#dcfce7">&#x1F49A;</div><div><h3>{{ stats.non_diabetic }}</h3><p>Non-Diabetic</p></div></div>
      <div class="sc"><div class="sc-i" style="background:#f3e8ff">&#x1F3C6;</div><div><h3>{{ best_acc }}%</h3><p>Best Accuracy</p></div></div>
      <div class="sc"><div class="sc-i" style="background:#fef3c7">&#x1F4CA;</div><div><h3>{{ stats.diabetic_pct }}%</h3><p>Prevalence</p></div></div>
      <div class="sc"><div class="sc-i" style="background:#e0f2fe">&#x1F9EA;</div><div><h3>{{ dtotal }}</h3><p>Platform Predictions</p></div></div>
    </div>
    <div class="cg2">
      <div class="cc"><div class="cc-t">Model Performance</div><canvas id="dr-chart" height="240"></canvas></div>
      <div class="cc"><div class="cc-t">Class Distribution</div><img src="data:image/png;base64,{{ charts.class_dist }}" class="ci"></div>
    </div>
    <div class="cc">
      <div class="cc-t">Model Metrics Comparison</div>
      <div style="overflow-x:auto"><table class="mt-t">
        <thead><tr><th>Model</th><th>Accuracy</th><th>F1 Score</th><th>Precision</th><th>Recall</th></tr></thead>
        <tbody>
        {% for nm,m in metrics.items() %}
        <tr {% if nm==best %}class="bst"{% endif %}>
          <td style="font-weight:600">{{ nm }}{% if nm==best %} <span style="background:var(--teal);color:var(--navy);font-size:.68rem;padding:.1rem .45rem;border-radius:50px;margin-left:.4rem">Best</span>{% endif %}</td>
          <td><div style="display:flex;align-items:center;gap:.5rem">{{ m.accuracy }}%<div class="pw"><div class="pf" style="width:{{ m.accuracy }}%"></div></div></div></td>
          <td>{{ m.f1 }}%</td><td>{{ m.precision }}%</td><td>{{ m.recall }}%</td>
        </tr>
        {% endfor %}
        </tbody>
      </table></div>
    </div>
  </div>

  <div class="tab-pane" id="pane-patients">
    <div class="cc"><div class="cc-t">All Patient Predictions</div>
      <div style="overflow-x:auto"><table class="pt">
        <thead><tr><th>#</th><th>Time</th><th>Age</th><th>BMI</th><th>Glucose</th><th>HbA1c</th><th>Best Model</th><th>Result</th><th>Risk %</th></tr></thead>
        <tbody id="pt-body"><tr><td colspan="9" style="text-align:center;color:var(--muted);padding:2rem">Loading...</td></tr></tbody>
      </table></div>
    </div>
  </div>

  <div class="tab-pane" id="pane-analytics">
    <div class="cg2">
      <div class="cc"><div class="cc-t">Age Distribution</div><img src="data:image/png;base64,{{ charts.age }}" class="ci"></div>
      <div class="cc"><div class="cc-t">BMI vs Diabetes</div><img src="data:image/png;base64,{{ charts.bmi }}" class="ci"></div>
    </div>
    <div class="cg2">
      <div class="cc"><div class="cc-t">Blood Glucose Distribution</div><img src="data:image/png;base64,{{ charts.glucose }}" class="ci"></div>
      <div class="cc"><div class="cc-t">HbA1c Distribution</div><img src="data:image/png;base64,{{ charts.hba1c }}" class="ci"></div>
    </div>
    <div class="cc"><div class="cc-t">Smoking History vs Diabetes</div><img src="data:image/png;base64,{{ charts.smoke }}" class="ci"></div>
    <div class="cg2">
      <div class="cc"><div class="cc-t">Hypertension &amp; Heart Disease</div><img src="data:image/png;base64,{{ charts.risk_factors }}" class="ci"></div>
      <div class="cc"><div class="cc-t">Feature Importance</div><img src="data:image/png;base64,{{ charts.feat_imp }}" class="ci"></div>
    </div>
    <div class="cc"><div class="cc-t">ROC Curves</div><img src="data:image/png;base64,{{ charts.roc }}" class="ci"></div>
    <div class="cg2">
      <div class="cc"><div class="cc-t">Correlation Heatmap</div><img src="data:image/png;base64,{{ charts.corr }}" class="ci"></div>
      <div class="cc"><div class="cc-t">Age vs BMI Scatter</div><img src="data:image/png;base64,{{ charts.age_bmi }}" class="ci"></div>
    </div>
  </div>

  <div class="tab-pane" id="pane-models">
    <div class="cg2">
      <div class="cc"><div class="cc-t">Random Forest CM</div><img src="data:image/png;base64,{{ charts.cm_random_forest }}" class="ci"></div>
      <div class="cc"><div class="cc-t">Logistic Regression CM</div><img src="data:image/png;base64,{{ charts.cm_logistic_regression }}" class="ci"></div>
    </div>
    <div class="cg2">
      <div class="cc"><div class="cc-t">XGBoost CM</div><img src="data:image/png;base64,{{ charts.cm_xgboost }}" class="ci"></div>
      <div class="cc"><div class="cc-t">LightGBM CM</div><img src="data:image/png;base64,{{ charts.cm_lightgbm }}" class="ci"></div>
    </div>
    <div class="cg2">
      <div class="cc"><div class="cc-t">Decision Tree CM</div><img src="data:image/png;base64,{{ charts.cm_decision_tree }}" class="ci"></div>
      <div class="cc"><div class="cc-t">PR Curve</div><img src="data:image/png;base64,{{ charts.pr_curve }}" class="ci"></div>
    </div>
    <div class="cc"><div class="cc-t">SHAP Summary</div><img src="data:image/png;base64,{{ charts.shap_summary }}" class="ci"></div>
    <div class="cg2">
    {% for nm,m in metrics.items() %}
      <div class="cc"><div class="cc-t">{{ nm }} &mdash; Confusion Details</div>
        <table style="width:100%;font-size:.85rem;border-collapse:collapse">
          <tr><td style="padding:.35rem;color:var(--muted)">True Positives</td><td style="font-weight:700;color:#16a34a">{{ m.tp }}</td></tr>
          <tr><td style="padding:.35rem;color:var(--muted)">True Negatives</td><td style="font-weight:700;color:#16a34a">{{ m.tn }}</td></tr>
          <tr><td style="padding:.35rem;color:var(--muted)">False Positives</td><td style="font-weight:700;color:#dc2626">{{ m.fp }}</td></tr>
          <tr><td style="padding:.35rem;color:var(--muted)">False Negatives</td><td style="font-weight:700;color:#dc2626">{{ m.fn }}</td></tr>
        </table>
      </div>
    {% endfor %}
    </div>
  </div>

  <div class="tab-pane" id="pane-literature">
    {% for color,ico,title,body in lit_items %}
    <div style="background:#fff;border-radius:12px;padding:1.6rem;box-shadow:0 2px 16px rgba(0,0,0,.07);border-left:4px solid {{ color }};margin-bottom:1rem">
      <h3 style="font-family:'Sora',sans-serif;font-weight:700;margin-bottom:.5rem">{{ ico }} {{ title }}</h3>
      <p style="color:var(--muted);font-size:.87rem;line-height:1.65">{{ body }}</p>
    </div>
    {% endfor %}
  </div>
</div>

<script>
const MM = {{ metrics|tojson }};
const BM = {{ best|tojson }};

function switchTab(id, el){
  document.querySelectorAll('.tab-pane').forEach(p=>p.classList.remove('active'));
  document.querySelectorAll('.tab-btn').forEach(b=>b.classList.remove('active'));
  document.getElementById('pane-'+id).classList.add('active');
  el.classList.add('active');
  if(id==='patients') loadPats();
}

new Chart(document.getElementById('dr-chart').getContext('2d'),{
  type:'bar',
  data:{labels:Object.keys(MM),datasets:[
    {label:'Accuracy %',data:Object.values(MM).map(m=>m.accuracy),backgroundColor:'rgba(0,212,170,.82)',borderRadius:5},
    {label:'F1 Score %',data:Object.values(MM).map(m=>m.f1),backgroundColor:'rgba(109,40,217,.72)',borderRadius:5},
    {label:'Precision %',data:Object.values(MM).map(m=>m.precision),backgroundColor:'rgba(14,165,233,.72)',borderRadius:5},
    {label:'Recall %',data:Object.values(MM).map(m=>m.recall),backgroundColor:'rgba(245,158,11,.72)',borderRadius:5},
  ]},
  options:{responsive:true,maintainAspectRatio:true,
    plugins:{legend:{position:'bottom',labels:{font:{size:10}}}},
    scales:{y:{beginAtZero:true,max:105,ticks:{callback:v=>v+'%',font:{size:10}}},
            x:{grid:{display:false},ticks:{font:{size:10}}}}}
});

async function loadPats(){
  try{
    const r = await fetch('/doctor/pred_log');
    const data = await r.json();
    const tb = document.getElementById('pt-body');
    if(!data.length){tb.innerHTML='<tr><td colspan="9" style="text-align:center;color:var(--muted);padding:2rem">No patient predictions yet.</td></tr>';return;}
    tb.innerHTML = data.map((p,i)=>`<tr>
      <td>${data.length-i}</td><td>${p.time}</td><td>${p.age}</td>
      <td>${p.bmi}</td><td>${p.glucose}</td><td>${p.hba1c}</td>
      <td style="font-size:.8rem">${p.best_model}</td>
      <td><span class="pill ${p.prediction===1?'pl-d':'pl-n'}">${p.prediction===1?'Diabetic':'Non-Diabetic'}</span></td>
      <td>${p.avg_risk}%</td></tr>`).join('');
  }catch(e){console.error(e);}
}
</script>
</body>
</html>"""


# =====================================================================
# ADMIN DASHBOARD HTML
# =====================================================================
ADMIN_DASH_HTML = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Admin Dashboard - DiaPredict</title>
<link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css" rel="stylesheet">
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<style>
:root{--navy:#060f1e;--navy2:#0c1a2e;--navy3:#0f2040;--teal:#00d4aa;--teal2:#00a896;
  --violet:#6d28d9;--bg:#f0f4f8;--card:#fff;--text:#0c1a2e;--muted:#5a6a7a;--border:#dde3ec;--r:12px;--sh:0 2px 16px rgba(0,0,0,.07)}
*{box-sizing:border-box;margin:0;padding:0}
body{font-family:'Plus Jakarta Sans',sans-serif;color:var(--text);background:var(--bg);display:flex;min-height:100vh}
a{text-decoration:none;color:inherit}
.sb{width:252px;background:var(--navy);position:fixed;top:0;left:0;height:100vh;display:flex;flex-direction:column;overflow-y:auto;z-index:100}
.sb-brand{padding:1.4rem 1.3rem;border-bottom:1px solid rgba(255,255,255,.07)}
.sb-logo{display:flex;align-items:center;gap:.6rem}
.sb-ico{width:36px;height:36px;background:linear-gradient(135deg,var(--teal),var(--violet));border-radius:9px;display:flex;align-items:center;justify-content:center;font-size:17px}
.sb-nm{font-family:'Sora',sans-serif;font-weight:800;font-size:1.05rem;background:linear-gradient(90deg,var(--teal),#a78bfa);-webkit-background-clip:text;-webkit-text-fill-color:transparent}
.sb-usr{padding:.88rem 1.3rem;border-bottom:1px solid rgba(255,255,255,.07);display:flex;align-items:center;gap:.7rem}
.sb-av{width:33px;height:33px;background:linear-gradient(135deg,var(--teal),var(--violet));border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:12px;font-weight:700;color:#fff;text-transform:uppercase}
.sb-un{color:#fff;font-size:.82rem;font-weight:600}
.sb-ur{color:rgba(255,255,255,.35);font-size:.67rem}
.sb-nav{padding:.8rem 0;flex:1}
.nsec{padding:.24rem 1.3rem .06rem;font-size:.62rem;font-weight:700;color:rgba(255,255,255,.2);text-transform:uppercase;letter-spacing:.1em;margin-top:.5rem}
.sl{display:flex;align-items:center;gap:.7rem;padding:.62rem 1.3rem;color:rgba(255,255,255,.48);font-size:.83rem;cursor:pointer;transition:.16s;border-left:3px solid transparent;border:none;background:none;width:100%;text-align:left}
.sl:hover{color:#fff;background:rgba(255,255,255,.05);border-left-color:rgba(0,212,170,.36)}
.sl.active{color:#fff;background:rgba(0,212,170,.1);border-left-color:var(--teal)}
.sl i{width:16px;text-align:center;font-size:.83rem}
.sb-ft{padding:.82rem 1.3rem;border-top:1px solid rgba(255,255,255,.07)}
.lo{display:flex;align-items:center;gap:.5rem;color:rgba(255,255,255,.4);font-size:.82rem;padding:.5rem .7rem;border-radius:7px;transition:.16s}
.lo:hover{background:rgba(220,38,38,.1);color:#fca5a5}
.mn{margin-left:252px;flex:1;display:flex;flex-direction:column}
.tb{background:#fff;padding:0 1.8rem;height:60px;display:flex;align-items:center;justify-content:space-between;border-bottom:1px solid var(--border);position:sticky;top:0;z-index:50}
.tb h1{font-family:'Sora',sans-serif;font-size:1.12rem;font-weight:700}
.apill{background:rgba(0,212,170,.1);color:#0f766e;border-radius:50px;padding:.25rem .78rem;font-size:.77rem;font-weight:600;display:flex;align-items:center;gap:.33rem}
.ct{padding:1.7rem;flex:1}
.tab{display:none}.tab.active{display:block}
.sg{display:grid;grid-template-columns:repeat(auto-fit,minmax(162px,1fr));gap:1rem;margin-bottom:1.7rem}
.sc{background:#fff;border-radius:12px;padding:1.15rem 1.25rem;box-shadow:var(--sh);border:1px solid var(--border);display:flex;align-items:center;gap:.82rem;transition:.2s}
.sc:hover{transform:translateY(-2px);box-shadow:0 7px 26px rgba(0,0,0,.09)}
.sc-i{width:43px;height:43px;border-radius:10px;display:flex;align-items:center;justify-content:center;font-size:1.22rem;flex-shrink:0}
.sc h3{font-family:'Sora',sans-serif;font-size:1.5rem;font-weight:800;line-height:1}
.sc p{font-size:.71rem;color:var(--muted);margin-top:.14rem}
.cc{background:#fff;border-radius:12px;padding:1.3rem;box-shadow:var(--sh);border:1px solid var(--border)}
.ct-t{font-family:'Sora',sans-serif;font-size:.92rem;font-weight:700;margin-bottom:.85rem;display:flex;align-items:center;gap:.4rem}
.cg{display:grid;grid-template-columns:repeat(auto-fit,minmax(310px,1fr));gap:1.3rem;margin-bottom:1.7rem}
.cg3{grid-template-columns:repeat(auto-fit,minmax(260px,1fr))}
.cg1{grid-template-columns:1fr}
.ci{width:100%;border-radius:7px}
.mt{width:100%;border-collapse:collapse;background:#fff;border-radius:12px;overflow:hidden;box-shadow:var(--sh)}
.mt th{background:var(--navy);color:rgba(255,255,255,.75);padding:.78rem 1.05rem;text-align:left;font-size:.72rem;font-weight:600;text-transform:uppercase;letter-spacing:.06em}
.mt td{padding:.78rem 1.05rem;border-bottom:1px solid var(--border);font-size:.86rem}
.mt tr:last-child td{border-bottom:none}
.mt tr:hover td{background:#f8fafc}
.mt tr.bst td{background:rgba(0,212,170,.055)}
.bdg{display:inline-flex;padding:.16rem .6rem;border-radius:50px;font-size:.7rem;font-weight:700}
.b-best{background:linear-gradient(135deg,var(--teal),#0097a7);color:var(--navy)}
.b-blue{background:rgba(59,130,246,.12);color:#1d4ed8}
.pt{width:100%;border-collapse:collapse;font-size:.82rem}
.pt th{background:#f8fafc;color:var(--muted);padding:.64rem .88rem;text-align:left;font-size:.71rem;font-weight:700;text-transform:uppercase;letter-spacing:.05em;border-bottom:1px solid var(--border)}
.pt td{padding:.64rem .88rem;border-bottom:1px solid var(--border)}
.pt tr:last-child td{border-bottom:none}
.pk{display:inline-block;padding:.17rem .57rem;border-radius:50px;font-size:.7rem;font-weight:600}
.pk-d{background:#fee2e2;color:#b91c1c}.pk-n{background:#dcfce7;color:#15803d}
.sh{display:flex;align-items:center;justify-content:space-between;margin-bottom:1.3rem}
.sh h2{font-family:'Sora',sans-serif;font-size:1.12rem;font-weight:700}
.pw{height:7px;background:#e5e7eb;border-radius:50px;overflow:hidden}
.pf{height:100%;border-radius:50px}
.dtw{overflow-x:auto}
.dt{width:100%;border-collapse:collapse;font-size:.78rem}
.dt th{background:var(--navy);color:rgba(255,255,255,.7);padding:.54rem .82rem;text-align:left;white-space:nowrap;font-size:.69rem;text-transform:uppercase;letter-spacing:.05em}
.dt td{padding:.54rem .82rem;border-bottom:1px solid var(--border);white-space:nowrap}
.dt tr:nth-child(even) td{background:#f8fafc}
.lc{background:#fff;border-radius:12px;padding:1.55rem;box-shadow:var(--sh);border-left:4px solid var(--teal);margin-bottom:1rem}
.lc h3{font-family:'Sora',sans-serif;font-weight:700;margin-bottom:.5rem;display:flex;align-items:center;gap:.4rem;font-size:.97rem}
.lc p{color:var(--muted);font-size:.84rem;line-height:1.65}
</style>
</head>
<body>
<aside class="sb">
  <div class="sb-brand">
    <div class="sb-logo">
      <div class="sb-ico">&#x1FA7A;</div>
      <div><div class="sb-nm">DiaPredict</div><div style="color:rgba(255,255,255,.3);font-size:.68rem">Admin Panel</div></div>
    </div>
  </div>
  <div class="sb-usr">
    <div class="sb-av">{{ auser[0] }}</div>
    <div><div class="sb-un">{{ auser }}</div><div class="sb-ur">System Admin</div></div>
  </div>
  <nav class="sb-nav">
    <div class="nsec">Main</div>
    <button class="sl active" onclick="goTab('ov',this)"><i class="fa fa-gauge-high"></i>Overview</button>
    <button class="sl" onclick="goTab('eda',this)"><i class="fa fa-chart-area"></i>EDA Charts</button>
    <button class="sl" onclick="goTab('mdl',this)"><i class="fa fa-robot"></i>Model Evaluation</button>
    <div class="nsec">Users</div>
    <button class="sl" onclick="goTab('users',this)"><i class="fa fa-users"></i>User Accounts</button>
    <button class="sl" onclick="goTab('log',this)"><i class="fa fa-list-check"></i>Prediction Log</button>
    <button class="sl" onclick="goTab('ds',this)"><i class="fa fa-table"></i>Dataset Preview</button>
    <div class="nsec">Docs</div>
    <button class="sl" onclick="goTab('lit',this)"><i class="fa fa-book-open"></i>Literature</button>
    <a class="sl" href="/" target="_blank"><i class="fa fa-arrow-up-right-from-square"></i>Public Site</a>
  </nav>
  <div class="sb-ft">
    <a href="/admin/logout" class="lo"><i class="fa fa-right-from-bracket"></i>Sign Out</a>
  </div>
</aside>

<div class="mn">
  <div class="tb">
    <h1 id="pg-t">Dashboard Overview</h1>
    <div style="display:flex;align-items:center;gap:.85rem">
      <div class="apill"><i class="fa fa-circle" style="font-size:.43rem;color:#16a34a"></i>All Systems Active</div>
      <div style="font-size:.77rem;color:var(--muted)" id="clk"></div>
    </div>
  </div>
  <div class="ct">
    <div class="tab active" id="tab-ov">
      <div class="sh"><div><h2>System Overview</h2><p style="color:var(--muted);font-size:.85rem">Live metrics from {{ stats.total }} records</p></div></div>
      <div class="sg">
        <div class="sc"><div class="sc-i" style="background:#dbeafe">&#128101;</div><div><h3>{{ stats.total }}</h3><p>Dataset Records</p></div></div>
        <div class="sc"><div class="sc-i" style="background:#fee2e2">&#x1F9AA;</div><div><h3>{{ stats.diabetic }}</h3><p>Diabetic ({{ stats.diabetic_pct }}%)</p></div></div>
        <div class="sc"><div class="sc-i" style="background:#dcfce7">&#x1F49A;</div><div><h3>{{ stats.non_diabetic }}</h3><p>Non-Diabetic</p></div></div>
        <div class="sc"><div class="sc-i" style="background:#f3e8ff">&#x1F3C6;</div><div><h3>{{ best_acc }}%</h3><p>Best Accuracy</p></div></div>
        <div class="sc"><div class="sc-i" style="background:#fef3c7">&#x1F464;</div><div><h3>{{ pat_cnt }}</h3><p>Registered Patients</p></div></div>
        <div class="sc"><div class="sc-i" style="background:#e0f2fe">&#x1FA7A;</div><div><h3>{{ dr_cnt }}</h3><p>Registered Doctors</p></div></div>
        <div class="sc"><div class="sc-i" style="background:#fff7ed">&#x1F4CA;</div><div><h3>{{ pred_cnt }}</h3><p>Total Predictions</p></div></div>
        <div class="sc"><div class="sc-i" style="background:#fdf4ff">&#x1F9EA;</div><div><h3>{{ stats.avg_hba1c }}</h3><p>Avg HbA1c</p></div></div>
      </div>
      <div class="cg">
        <div class="cc"><div class="ct-t"><i class="fa fa-chart-bar" style="color:var(--teal)"></i>Model Comparison</div><canvas id="aChart" height="235"></canvas></div>
        <div class="cc"><div class="ct-t">Class Distribution</div><img src="data:image/png;base64,{{ charts.class_dist }}" class="ci"></div>
      </div>
      <div class="cc" style="margin-bottom:1.7rem">
        <div class="ct-t" style="margin-bottom:1rem">Live Model Metrics</div>
        <table class="mt">
          <thead><tr><th>Model</th><th>Accuracy</th><th>F1 Score</th><th>Precision</th><th>Recall</th><th>Status</th></tr></thead>
          <tbody>
          {% for nm,m in metrics.items() %}
          <tr {% if nm==best %}class="bst"{% endif %}>
            <td style="font-weight:600">{{ nm }}</td>
            <td><div style="display:flex;align-items:center;gap:.55rem">{{ m.accuracy }}%<div class="pw" style="width:70px"><div class="pf" style="width:{{ m.accuracy }}%;background:var(--teal)"></div></div></div></td>
            <td>{{ m.f1 }}%</td><td>{{ m.precision }}%</td><td>{{ m.recall }}%</td>
            <td>{% if nm==best %}<span class="bdg b-best">&#11088; Best</span>{% else %}<span class="bdg b-blue">Active</span>{% endif %}</td>
          </tr>
          {% endfor %}
          </tbody>
        </table>
      </div>
    </div>

    <div class="tab" id="tab-eda">
      <div class="sh"><div><h2>Exploratory Analysis</h2></div></div>
      <div class="cg"><div class="cc"><div class="ct-t">Age Distribution</div><img src="data:image/png;base64,{{ charts.age }}" class="ci"></div><div class="cc"><div class="ct-t">BMI vs Diabetes</div><img src="data:image/png;base64,{{ charts.bmi }}" class="ci"></div></div>
      <div class="cg"><div class="cc"><div class="ct-t">Blood Glucose</div><img src="data:image/png;base64,{{ charts.glucose }}" class="ci"></div><div class="cc"><div class="ct-t">HbA1c Distribution</div><img src="data:image/png;base64,{{ charts.hba1c }}" class="ci"></div></div>
      <div class="cg cg1"><div class="cc"><div class="ct-t">Smoking History</div><img src="data:image/png;base64,{{ charts.smoke }}" class="ci"></div></div>
      <div class="cg"><div class="cc"><div class="ct-t">Age vs BMI</div><img src="data:image/png;base64,{{ charts.age_bmi }}" class="ci"></div><div class="cc"><div class="ct-t">Risk Factors</div><img src="data:image/png;base64,{{ charts.risk_factors }}" class="ci"></div></div>
      <div class="cg cg1"><div class="cc"><div class="ct-t">Correlation Heatmap</div><img src="data:image/png;base64,{{ charts.corr }}" class="ci"></div></div>
    </div>

    <div class="tab" id="tab-mdl">
      <div class="sh"><div><h2>Model Evaluation</h2></div></div>
      <div class="cg cg1"><div class="cc"><div class="ct-t">ROC Curves</div><img src="data:image/png;base64,{{ charts.roc }}" class="ci"></div></div>
      <div class="cg cg1"><div class="cc"><div class="ct-t">Feature Importance</div><img src="data:image/png;base64,{{ charts.feat_imp }}" class="ci"></div></div>
      <div class="cg cg3">
        <div class="cc"><div class="ct-t">Random Forest CM</div><img src="data:image/png;base64,{{ charts.cm_random_forest }}" class="ci"></div>
        <div class="cc"><div class="ct-t">Logistic Regression CM</div><img src="data:image/png;base64,{{ charts.cm_logistic_regression }}" class="ci"></div>
        <div class="cc"><div class="ct-t">XGBoost CM</div><img src="data:image/png;base64,{{ charts.cm_xgboost }}" class="ci"></div>
        <div class="cc"><div class="ct-t">LightGBM CM</div><img src="data:image/png;base64,{{ charts.cm_lightgbm }}" class="ci"></div>
        <div class="cc"><div class="ct-t">Decision Tree CM</div><img src="data:image/png;base64,{{ charts.cm_decision_tree }}" class="ci"></div>
        <div class="cc"><div class="ct-t">PR Curve</div><img src="data:image/png;base64,{{ charts.pr_curve }}" class="ci"></div>
        <div class="cc"><div class="ct-t">SHAP Summary</div><img src="data:image/png;base64,{{ charts.shap_summary }}" class="ci"></div>
      </div>
    </div>

    <div class="tab" id="tab-users">
      <div class="sh"><div><h2>User Accounts</h2></div></div>
      <div class="cc" style="margin-bottom:1.4rem">
        <div class="ct-t"><i class="fa fa-user" style="color:var(--teal)"></i>Registered Patients</div>
        <table class="pt"><thead><tr><th>Name</th><th>Email</th><th>Gender</th><th>Age</th><th>Joined</th><th>Assessments</th></tr></thead>
          <tbody id="pat-body"><tr><td colspan="6" style="text-align:center;color:var(--muted);padding:1.5rem">No patients yet.</td></tr></tbody></table>
      </div>
      <div class="cc">
        <div class="ct-t"><i class="fa fa-stethoscope" style="color:var(--violet)"></i>Registered Doctors</div>
        <table class="pt"><thead><tr><th>Name</th><th>Email</th><th>Specialty</th><th>Hospital</th><th>Joined</th></tr></thead>
          <tbody id="dr-body"><tr><td colspan="5" style="text-align:center;color:var(--muted);padding:1.5rem">No doctors yet.</td></tr></tbody></table>
      </div>
    </div>

    <div class="tab" id="tab-log">
      <div class="sh"><div><h2>Prediction Log</h2></div>
        <button onclick="loadLog()" style="background:var(--teal);color:var(--navy);border:none;padding:.43rem .85rem;border-radius:7px;font-weight:700;cursor:pointer;font-size:.82rem"><i class="fa fa-rotate"></i> Refresh</button>
      </div>
      <div style="overflow-x:auto"><table class="pt">
        <thead><tr><th>#</th><th>Time</th><th>Age</th><th>BMI</th><th>Glucose</th><th>HbA1c</th><th>Best Model</th><th>Result</th><th>Risk %</th></tr></thead>
        <tbody id="log-body"><tr><td colspan="9" style="text-align:center;color:var(--muted);padding:2rem">No predictions yet.</td></tr></tbody>
      </table></div>
    </div>

    <div class="tab" id="tab-ds">
      <div class="sh"><div><h2>Dataset Preview</h2></div></div>
      <div class="cc dtw"><div class="ct-t" style="margin-bottom:.85rem"><i class="fa fa-table"></i> First 15 Records</div>
        <table class="dt">
          <thead><tr>{% for c in df_cols %}<th>{{ c }}</th>{% endfor %}</tr></thead>
          <tbody>{% for row in df_preview %}<tr>{% for c in df_cols %}<td>{{ row[c] }}</td>{% endfor %}</tr>{% endfor %}</tbody>
        </table>
      </div>
    </div>

    <div class="tab" id="tab-lit">
      <div class="sh"><div><h2>Literature Review</h2></div></div>
      <div class="lc"><h3><i class="fa fa-tree" style="color:#22c55e"></i> Random Forest</h3><p>Ensemble of decision trees capturing complex non-linear relationships in clinical features. Provides feature importance rankings for transparent clinical insight. Robust to outliers and high-dimensional data.</p></div>
      <div class="lc" style="border-left-color:#3b82f6"><h3><i class="fa fa-magnifying-glass-chart" style="color:#3b82f6"></i> Logistic Regression</h3><p>Gold standard baseline for clinical predictive modeling. Interpretable coefficients directly quantify each feature's contribution to diabetes risk, making it ideal for clinical communication.</p></div>
      <div class="lc" style="border-left-color:#f59e0b"><h3><i class="fa fa-bolt" style="color:#f59e0b"></i> XGBoost &amp; LightGBM</h3><p>State-of-the-art gradient boosting with L1/L2 regularization, sequential error correction, and native missing-value handling. Top performers on structured medical datasets.</p></div>
      <div class="lc" style="border-left-color:var(--muted)">
        <h3><i class="fa fa-book"></i> References</h3>
        <div style="background:#f8fafc;border-radius:8px;padding:.95rem;margin-top:.65rem">
          <p style="font-size:.76rem;color:var(--muted);line-height:1.8">
            &bull; Ali et al. (2019). Early diagnosis using ML. <em>Knowledge-Based Systems, 163</em>, 12&ndash;25.<br>
            &bull; Kavakiotis et al. (2017). ML in diabetes research. <em>Comp. Struct. Biotechnol. J, 15</em>, 104&ndash;116.<br>
            &bull; Diabetes Prevention Program (2002). <em>NEJM, 346</em>(6), 393&ndash;403.
          </p>
        </div>
      </div>
    </div>
  </div>
</div>

<script>
const MM={{ metrics|tojson }},BM={{ best|tojson }};
const TITLES={ov:'Dashboard Overview',eda:'EDA Charts',mdl:'Model Evaluation',
              users:'User Accounts',log:'Prediction Log',ds:'Dataset Preview',lit:'Literature'};
setInterval(()=>{document.getElementById('clk').textContent=new Date().toLocaleTimeString('en-US',{hour:'2-digit',minute:'2-digit',second:'2-digit'});},1000);

function goTab(id,el){
  document.querySelectorAll('.tab').forEach(t=>t.classList.remove('active'));
  document.querySelectorAll('.sl').forEach(l=>l.classList.remove('active'));
  document.getElementById('tab-'+id).classList.add('active');
  document.getElementById('pg-t').textContent=TITLES[id];
  if(el)el.classList.add('active');
  if(id==='log')loadLog();
  if(id==='users')loadUsers();
}

new Chart(document.getElementById('aChart').getContext('2d'),{type:'bar',
  data:{labels:Object.keys(MM),datasets:[
    {label:'Accuracy %',data:Object.values(MM).map(m=>m.accuracy),backgroundColor:'rgba(0,212,170,.82)',borderRadius:5},
    {label:'F1 %',data:Object.values(MM).map(m=>m.f1),backgroundColor:'rgba(109,40,217,.72)',borderRadius:5},
    {label:'Precision %',data:Object.values(MM).map(m=>m.precision),backgroundColor:'rgba(14,165,233,.72)',borderRadius:5},
    {label:'Recall %',data:Object.values(MM).map(m=>m.recall),backgroundColor:'rgba(245,158,11,.72)',borderRadius:5},
  ]},
  options:{responsive:true,maintainAspectRatio:true,
    plugins:{legend:{position:'bottom',labels:{font:{size:10}}}},
    scales:{y:{beginAtZero:true,max:105,ticks:{callback:v=>v+'%',font:{size:10}},grid:{color:'rgba(0,0,0,.04)'}},
            x:{grid:{display:false},ticks:{font:{size:10}}}}}
});

async function loadLog(){
  try{const r=await fetch('/admin/pred_log');const data=await r.json();
    const tb=document.getElementById('log-body');
    if(!data.length){tb.innerHTML='<tr><td colspan="9" style="text-align:center;color:var(--muted);padding:2rem">No predictions yet.</td></tr>';return;}
    tb.innerHTML=data.map((p,i)=>`<tr><td>${data.length-i}</td><td>${p.time}</td><td>${p.age}</td><td>${p.bmi}</td><td>${p.glucose}</td><td>${p.hba1c}</td><td style="font-size:.78rem">${p.best_model}</td><td><span class="pk ${p.prediction===1?'pk-d':'pk-n'}">${p.prediction===1?'Diabetic':'Non-Diabetic'}</span></td><td>${p.avg_risk}%</td></tr>`).join('');
  }catch(e){console.error(e);}
}

async function loadUsers(){
  try{const r=await fetch('/admin/users');const data=await r.json();
    const pb=document.getElementById('pat-body');const db=document.getElementById('dr-body');
    if(data.patients.length){pb.innerHTML=data.patients.map(p=>`<tr><td>${p.name}</td><td>${p.email}</td><td>${p.gender}</td><td>${p.age}</td><td>${p.joined}</td><td>${p.assessments}</td></tr>`).join('');}
    else pb.innerHTML='<tr><td colspan="6" style="text-align:center;color:var(--muted);padding:1.5rem">No patients yet.</td></tr>';
    if(data.doctors.length){db.innerHTML=data.doctors.map(d=>`<tr><td>${d.name}</td><td>${d.email}</td><td>${d.specialty}</td><td>${d.hospital}</td><td>${d.joined}</td></tr>`).join('');}
    else db.innerHTML='<tr><td colspan="5" style="text-align:center;color:var(--muted);padding:1.5rem">No doctors yet.</td></tr>';
  }catch(e){console.error(e);}
}
</script>
</body>
</html>"""


# =====================================================================
# FLASK APP & ROUTES
# =====================================================================
app = Flask(__name__)
app.secret_key = 'DiaPredict-2026-SuperSecret-X9K!'
ADMINS = {
    'admin':        hash_pw('admin123'),
    'doctor_admin': hash_pw('doctor2026'),
}
LIT_ITEMS = [
    ('#22c55e', '&#x1F332;', 'Random Forest',
     'Ensemble of decision trees capturing non-linear relationships in clinical features. Feature importance rankings deliver transparent insight.'),
    ('#3b82f6', '&#x1F4C8;', 'Logistic Regression',
     'Gold standard clinical baseline providing fully interpretable coefficients. Ideal for communicating risk factors to patients.'),
    ('#f59e0b', '&#x26A1;', 'XGBoost',
     'State-of-the-art gradient boosting with L1/L2 regularization and native missing-value handling. Top performer on structured medical datasets.'),
    ('#ef4444', '&#x1F3E5;', 'Clinical Applications',
     'Early ML-based diabetes screening can reduce Type 2 onset by up to 58% through timely lifestyle interventions (Diabetes Prevention Program, NEJM 2002).'),
]


@app.route('/')
def home():
    return render_template_string(HOME_HTML,
        stats=stats, best=BEST_MODEL, best_acc=BEST_ACC, metrics=model_metrics)


@app.route('/patient/signup', methods=['GET', 'POST'])
def patient_signup():
    error = None
    if request.method == 'POST':
        name   = request.form.get('name', '').strip()
        email  = request.form.get('email', '').strip().lower()
        pw     = request.form.get('password', '')
        pw2    = request.form.get('password2', '')
        age    = request.form.get('age', '')
        gender = request.form.get('gender', '')
        phone  = request.form.get('phone', '').strip()
        if not all([name, email, pw, age, gender]):
            error = 'All required fields must be filled.'
        elif not valid_email(email):
            error = 'Please enter a valid email address.'
        elif email in patients_db:
            error = 'An account with this email already exists.'
        elif pw != pw2:
            error = 'Passwords do not match.'
        elif len(pw) < 6:
            error = 'Password must be at least 6 characters.'
        else:
            patients_db[email] = {'name': name, 'pw': hash_pw(pw), 'age': age,
                                   'gender': gender, 'phone': phone,
                                   'joined': datetime.datetime.now().strftime('%Y-%m-%d')}
            patient_hist[email] = []
            session['patient'] = email
            return redirect('/patient/dashboard')
    raw_form = (
        (f'<div class="err-msg"><i class="fa fa-circle-xmark"></i> {error}</div>' if error else '') +
        """<form method="POST" action="/patient/signup">
          <div class="fgrid2">
            <div class="frow"><label>Full Name *</label><div class="fw"><i class="fa fa-user"></i><input name="name" type="text" placeholder="Your full name" required></div></div>
            <div class="frow"><label>Age *</label><div class="fw"><i class="fa fa-calendar"></i><input name="age" type="number" min="1" max="120" placeholder="Age" required></div></div>
          </div>
          <div class="frow"><label>Email Address *</label><div class="fw"><i class="fa fa-envelope"></i><input name="email" type="email" placeholder="your@email.com" required></div></div>
          <div class="frow"><label>Gender *</label><div class="fw"><i class="fa fa-venus-mars"></i><select name="gender" required><option value="" disabled selected>Select gender</option><option>Male</option><option>Female</option><option>Other</option></select></div></div>
          <div class="frow"><label>Phone (optional)</label><div class="fw"><i class="fa fa-phone"></i><input name="phone" type="tel" placeholder="+1 234 567 8900"></div></div>
          <div class="fgrid2">
            <div class="frow"><label>Password *</label><div class="fw"><i class="fa fa-lock"></i><input name="password" type="password" placeholder="Min 6 chars" required></div></div>
            <div class="frow"><label>Confirm Password *</label><div class="fw"><i class="fa fa-lock"></i><input name="password2" type="password" placeholder="Repeat" required></div></div>
          </div>
          <button type="submit" class="auth-btn"><i class="fa fa-user-plus"></i> Create Patient Account</button>
        </form>"""
    )
    return auth_page('Patient Sign Up', 'Create your free account to check diabetes risk', '&#x1F9D1;',
                     raw_form, '<div class="auth-foot">Already have an account? <a href="/patient/login">Sign In</a></div>')


@app.route('/patient/login', methods=['GET', 'POST'])
def patient_login():
    error = None
    if request.method == 'POST':
        email = request.form.get('email', '').strip().lower()
        pw    = request.form.get('password', '')
        if email in patients_db and patients_db[email]['pw'] == hash_pw(pw):
            session['patient'] = email
            return redirect('/patient/dashboard')
        error = 'Invalid email or password.'
    raw_form = (
        (f'<div class="err-msg"><i class="fa fa-circle-xmark"></i> {error}</div>' if error else '') +
        """<form method="POST" action="/patient/login">
          <div class="frow"><label>Email Address</label><div class="fw"><i class="fa fa-envelope"></i><input name="email" type="email" placeholder="your@email.com" required></div></div>
          <div class="frow"><label>Password</label><div class="fw"><i class="fa fa-lock"></i><input name="password" type="password" placeholder="Your password" required></div></div>
          <button type="submit" class="auth-btn"><i class="fa fa-right-to-bracket"></i> Sign In to Patient Portal</button>
        </form>
        <div class="divider"><hr><span>New here?</span><hr></div>
        <a href="/patient/signup" style="display:block;text-align:center;padding:.8rem;border:1.5px solid rgba(0,212,170,.3);border-radius:9px;color:var(--teal);font-weight:600;font-size:.9rem"><i class="fa fa-user-plus"></i> Create Free Account</a>"""
    )
    return auth_page('Patient Login', 'Sign in to access your diabetes risk portal', '&#x1FA7A;',
                     raw_form, '<div class="auth-foot"><a href="/">&#x2190; Back to Home</a></div>')

@app.route('/patient/dashboard')
def patient_dashboard():
    try:
        email = session.get('patient')

        # No patient session
        if not email:
            return redirect('/patient/login')

        # Session exists but patient account no longer exists
        if email not in patients_db:
            session.pop('patient', None)
            return redirect('/patient/login')

        pdata = patients_db[email]

        # Make sure history always exists
        patient_hist.setdefault(email, [])
        history = patient_hist[email]

        return render_template_string(
            PATIENT_DASHBOARD_HTML,
            pname=pdata.get('name', 'Patient'),
            page=pdata.get('age', 30),
            history=history,
            genders=get_genders(),
            smokes=get_smokes(),
            metrics=model_metrics,
            best=BEST_MODEL,
            global_fi=global_fi
        )

    except Exception as e:
        import traceback

        print("\n========== PATIENT DASHBOARD ERROR ==========")
        print(str(e))
        traceback.print_exc()
        print("=============================================\n")

        return f"""
        <html>
        <body style="font-family:Arial;padding:40px">
            <h2>Patient Dashboard Error</h2>
            <pre>{str(e)}</pre>
            <hr>
            <a href="/patient/login">Back to Patient Login</a>
        </body>
        </html>
        """, 500


@app.route('/patient/logout')
def patient_logout():
    session.pop('patient', None)
    return redirect('/')


@app.route('/doctor/signup', methods=['GET', 'POST'])
def doctor_signup():
    error = None
    if request.method == 'POST':
        name     = request.form.get('name', '').strip()
        email    = request.form.get('email', '').strip().lower()
        pw       = request.form.get('password', '')
        pw2      = request.form.get('password2', '')
        spec     = request.form.get('specialty', '').strip()
        hospital = request.form.get('hospital', '').strip()
        if not all([name, email, pw, spec, hospital]):
            error = 'All required fields must be filled.'
        elif not valid_email(email):
            error = 'Please enter a valid email address.'
        elif email in doctors_db:
            error = 'A doctor account with this email already exists.'
        elif pw != pw2:
            error = 'Passwords do not match.'
        elif len(pw) < 6:
            error = 'Password must be at least 6 characters.'
        else:
            doctors_db[email] = {'name': name, 'pw': hash_pw(pw), 'specialty': spec,
                                  'hospital': hospital, 'joined': datetime.datetime.now().strftime('%Y-%m-%d')}
            session['doctor'] = email
            return redirect('/doctor/dashboard')
    raw_form = (
        (f'<div class="err-msg"><i class="fa fa-circle-xmark"></i> {error}</div>' if error else '') +
        """<form method="POST" action="/doctor/signup">
          <div class="fgrid2">
            <div class="frow"><label>Full Name *</label><div class="fw"><i class="fa fa-user-md"></i><input name="name" type="text" placeholder="Dr. Full Name" required></div></div>
            <div class="frow"><label>Email Address *</label><div class="fw"><i class="fa fa-envelope"></i><input name="email" type="email" placeholder="doctor@hospital.com" required></div></div>
          </div>
          <div class="frow"><label>Medical Specialty *</label><div class="fw"><i class="fa fa-stethoscope"></i><select name="specialty" required><option value="" disabled selected>Select specialty</option><option>Endocrinology</option><option>Internal Medicine</option><option>General Practice</option><option>Cardiology</option><option>Family Medicine</option><option>Other</option></select></div></div>
          <div class="frow"><label>Hospital / Clinic *</label><div class="fw"><i class="fa fa-hospital"></i><input name="hospital" type="text" placeholder="Hospital or clinic name" required></div></div>
          <div class="fgrid2">
            <div class="frow"><label>Password *</label><div class="fw"><i class="fa fa-lock"></i><input name="password" type="password" placeholder="Min 6 chars" required></div></div>
            <div class="frow"><label>Confirm Password *</label><div class="fw"><i class="fa fa-lock"></i><input name="password2" type="password" placeholder="Repeat" required></div></div>
          </div>
          <button type="submit" class="auth-btn" style="background:linear-gradient(135deg,#6d28d9,#4c1d95)"><i class="fa fa-stethoscope"></i> Create Doctor Account</button>
        </form>"""
    )
    return auth_page('Doctor Sign Up', 'Join DiaPredict as a healthcare professional', '&#x1FA7A;',
                     raw_form, '<div class="auth-foot">Already registered? <a href="/doctor/login">Sign In</a></div>',
                     color='#6d28d9')


@app.route('/doctor/login', methods=['GET', 'POST'])
def doctor_login():
    error = None
    if request.method == 'POST':
        email = request.form.get('email', '').strip().lower()
        pw    = request.form.get('password', '')
        if email in doctors_db and doctors_db[email]['pw'] == hash_pw(pw):
            session['doctor'] = email
            return redirect('/doctor/dashboard')
        error = 'Invalid email or password.'
    raw_form = (
        (f'<div class="err-msg"><i class="fa fa-circle-xmark"></i> {error}</div>' if error else '') +
        """<form method="POST" action="/doctor/login">
          <div class="frow"><label>Doctor Email</label><div class="fw"><i class="fa fa-envelope"></i><input name="email" type="email" placeholder="doctor@hospital.com" required></div></div>
          <div class="frow"><label>Password</label><div class="fw"><i class="fa fa-lock"></i><input name="password" type="password" placeholder="Your password" required></div></div>
          <button type="submit" class="auth-btn" style="background:linear-gradient(135deg,#6d28d9,#4c1d95)"><i class="fa fa-stethoscope"></i> Sign In to Doctor Portal</button>
        </form>
        <div class="divider"><hr><span>Not registered?</span><hr></div>
        <a href="/doctor/signup" style="display:block;text-align:center;padding:.8rem;border:1.5px solid rgba(109,40,217,.4);border-radius:9px;color:#a78bfa;font-weight:600;font-size:.9rem"><i class="fa fa-user-plus"></i> Register as a Doctor</a>"""
    )
    return auth_page('Doctor Login', 'Access the clinical prediction portal', '&#x1FA7A;',
                     raw_form, '<div class="auth-foot"><a href="/">&#x2190; Back to Home</a></div>',
                     color='#6d28d9')


@app.route('/doctor/dashboard')
def doctor_dashboard():
    if 'doctor' not in session:
        return redirect('/doctor/login')
    email = session['doctor']
    ddata = doctors_db.get(email, {})
    return render_template_string(
        DOCTOR_DASH_HTML,
        dname=ddata.get('name', 'Doctor'), dspec=ddata.get('specialty', ''),
        dhosp=ddata.get('hospital', ''), dtotal=len(pred_log),
        stats=stats, charts=charts, metrics=model_metrics,
        best=BEST_MODEL, best_acc=BEST_ACC, lit_items=LIT_ITEMS,
    )


@app.route('/doctor/pred_log')
def doctor_pred_log():
    if 'doctor' not in session:
        return jsonify([])
    return jsonify(list(reversed(pred_log)))


@app.route('/doctor/logout')
def doctor_logout():
    session.pop('doctor', None)
    return redirect('/')


@app.route('/doctor/export_csv')
def doctor_export_csv():
    if 'doctor' not in session:
        return redirect('/doctor/login')
    si = io.StringIO()
    cw = csv.writer(si)
    cw.writerow(['Time', 'Age', 'BMI', 'Glucose', 'HbA1c',
                 'Best Model', 'Prediction', 'Risk %', 'Risk Level'])
    for p in reversed(pred_log):
        cw.writerow([p.get('time',''), p.get('age',''), p.get('bmi',''),
                     p.get('glucose',''), p.get('hba1c',''), p.get('best_model',''),
                     'Diabetic' if p.get('prediction')==1 else 'Non-Diabetic',
                     p.get('avg_risk',''), p.get('risk_level','')])
    return Response(si.getvalue(), mimetype='text/csv',
                    headers={'Content-Disposition': 'attachment;filename=DiaPredict_Patients.csv'})


@app.route('/predict', methods=['POST'])
def predict():
    if 'patient' not in session:
        return jsonify({'success': False, 'error': 'Not authenticated'})
    try:
        d = request.get_json(force=True)

        # FIX: gender and smoking_history come as class label strings from select dropdowns
        gender_val = le_gender.transform([d['gender']])[0]
        smoke_val  = le_smoke.transform([d['smoking_history']])[0]

        row = pd.DataFrame([[
            gender_val,
            float(d['age']),
            int(d['hypertension']),
            int(d['heart_disease']),
            smoke_val,
            float(d['bmi']),
            float(d['HbA1c_level']),          # FIX: consistent field name
            float(d['blood_glucose_level']),
        ]], columns=FEAT_COLS)

        xs = scaler.transform(row)

        results = {}
        for name, mdl in MODELS.items():
            pred = int(mdl.predict(xs)[0])
            prob = round(float(mdl.predict_proba(xs)[0][1]) * 100, 2)
            results[name] = {'prediction': pred, 'probability': prob,
                             'label': 'Diabetic' if pred == 1 else 'Non-Diabetic'}

        avg_risk = round(sum(r['probability'] for r in results.values()) / len(results), 2)

        if avg_risk < 30:
            risk_lvl = 'Low Risk'
            rec = 'Your risk appears low. Maintain a healthy lifestyle with regular exercise and a balanced diet. Monitor annually.'
        elif avg_risk < 60:
            risk_lvl = 'Moderate Risk'
            rec = 'Your risk is moderate. Consider consulting a doctor and monitoring your blood sugar regularly. Lifestyle improvements are recommended.'
        else:
            risk_lvl = 'High Risk'
            rec = 'URGENT: Consult a healthcare provider immediately. A medical evaluation is strongly recommended.'

        # FIX: LightGBM binary shap returns list of 2 arrays; use positive class [1]
        sv_raw = shap_explainer.shap_values(xs)
        if isinstance(sv_raw, list):
            sv = sv_raw[1][0]
        else:
            sv = sv_raw[0]

        shap_data = sorted(
            [{'feature': f, 'value': float(v)} for f, v in zip(FEAT_COLS, sv)],
            key=lambda x: abs(x['value']), reverse=True
        )

        email = session['patient']
        entry = {
            'time':           datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'age':            int(float(d['age'])),
            'bmi':            round(float(d['bmi']), 1),
            'glucose':        int(float(d['blood_glucose_level'])),
            'hba1c':          round(float(d['HbA1c_level']), 1),
            'best_model':     BEST_MODEL,
            'prediction':     results[BEST_MODEL]['prediction'],
            'confidence':     results[BEST_MODEL]['probability'],
            'avg_risk':       avg_risk,
            'risk_level':     risk_lvl,
            'recommendation': rec,
        }
        patient_hist.setdefault(email, []).append(entry)
        pred_log.append(entry)
        if len(pred_log) > 500:
            pred_log.pop(0)

        return jsonify({'success': True, 'results': results, 'best_model': BEST_MODEL,
                        'model_accuracies': model_metrics, 'avg_risk': avg_risk,
                        'risk_level': risk_lvl, 'recommendation': rec,
                        'shap_data': shap_data, 'global_fi': global_fi})
    except Exception as e:
        import traceback
        return jsonify({'success': False, 'error': str(e) + '\n' + traceback.format_exc()})


@app.route('/download_pdf')
def download_pdf():
    risk  = request.args.get('risk', 'N/A')
    level = request.args.get('level', 'N/A')
    rec   = request.args.get('rec', 'N/A')
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font('Arial', 'B', 16)
    pdf.cell(0, 12, 'DiaPredict Risk Assessment Report', ln=True, align='C')
    pdf.set_font('Arial', '', 12)
    pdf.ln(4)
    pdf.cell(0, 10, f'Date: {datetime.datetime.now().strftime("%Y-%m-%d")}', ln=True)
    pdf.cell(0, 10, f'Overall Risk Score: {risk}%', ln=True)
    pdf.cell(0, 10, f'Risk Level: {level}', ln=True)
    pdf.ln(4)
    pdf.set_font('Arial', 'B', 12)
    pdf.cell(0, 10, 'Recommendation:', ln=True)
    pdf.set_font('Arial', '', 11)
    # FIX: encode to latin-1 safe (FPDF default font limitation)
    safe_rec = rec.encode('latin-1', 'replace').decode('latin-1')
    pdf.multi_cell(0, 9, safe_rec)
    pdf.ln(8)
    pdf.set_font('Arial', 'B', 11)
    pdf.cell(0, 10, 'Medical Disclaimer:', ln=True)
    pdf.set_font('Arial', '', 10)
    pdf.multi_cell(0, 9,
        'This report is generated by Machine Learning model for informational purposes only. '
        'It does not constitute medical advice. Please consult a qualified healthcare '
        'provider for proper diagnosis and treatment.')
    pdf_path = '/tmp/diabetes_report.pdf'
    pdf.output(pdf_path)
    return send_file(pdf_path, as_attachment=True,
                     download_name='DiaPredict_Report.pdf',
                     mimetype='application/pdf')


@app.route('/admin')
def admin_home():
    return redirect('/admin/login')


@app.route('/admin/login', methods=['GET', 'POST'])
def admin_login():
    error = None
    if request.method == 'POST':
        u = request.form.get('username', '').strip()
        p = request.form.get('password', '')
        if u in ADMINS and ADMINS[u] == hash_pw(p):
            session['admin'] = u
            return redirect('/admin/dashboard')
        error = 'Invalid admin credentials.'
    raw_form = (
        (f'<div class="err-msg"><i class="fa fa-circle-xmark"></i> {error}</div>' if error else '') +
        """<form method="POST" action="/admin/login">
          <div class="frow"><label>Admin Username</label><div class="fw"><i class="fa fa-user-shield"></i><input name="username" type="text" placeholder="admin" required autocomplete="off"></div></div>
          <div class="frow"><label>Password</label><div class="fw"><i class="fa fa-lock"></i><input name="password" type="password" placeholder="Admin password" required></div></div>
          <button type="submit" class="auth-btn" style="background:linear-gradient(135deg,#dc2626,#991b1b)"><i class="fa fa-right-to-bracket"></i> Access Admin Panel</button>
        </form>"""
    )
    return auth_page('Admin Login', 'Restricted - authorized personnel only', '&#x1F510;',
                     raw_form, '<div class="auth-foot"><a href="/">&#x2190; Back to Home</a></div>',
                     color='#dc2626')


@app.route('/admin/dashboard')
def admin_dashboard():
    if 'admin' not in session:
        return redirect('/admin/login')
    return render_template_string(
        ADMIN_DASH_HTML,
        charts=charts, metrics=model_metrics, best=BEST_MODEL, best_acc=BEST_ACC,
        stats=stats, df_preview=df_preview, df_cols=df_cols,
        auser=session['admin'],
        pat_cnt=len(patients_db), dr_cnt=len(doctors_db), pred_cnt=len(pred_log),
    )


@app.route('/admin/pred_log')
def admin_pred_log():
    if 'admin' not in session:
        return jsonify([])
    return jsonify(list(reversed(pred_log)))


@app.route('/admin/users')
def admin_users():
    if 'admin' not in session:
        return jsonify({'patients': [], 'doctors': []})
    pats = [{'name': v['name'], 'email': k, 'gender': v.get('gender', ''),
              'age': v.get('age', ''), 'joined': v.get('joined', ''),
              'assessments': len(patient_hist.get(k, []))} for k, v in patients_db.items()]
    docs = [{'name': v['name'], 'email': k, 'specialty': v.get('specialty', ''),
              'hospital': v.get('hospital', ''), 'joined': v.get('joined', '')} for k, v in doctors_db.items()]
    return jsonify({'patients': pats, 'doctors': docs})


@app.route('/admin/logout')
def admin_logout():
    session.pop('admin', None)
    return redirect('/')


# =====================================================================
# START FLASK + NGROK
# =====================================================================
def run_flask():
    app.run(host='0.0.0.0', port=5050, debug=False, use_reloader=False)


flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(3)

try:
    ngrok.kill()
    time.sleep(1)
except Exception:
    pass

tunnel     = ngrok.connect(5050)
public_url = tunnel.public_url

print("\n" + "=" * 65)
print(f"  Homepage       ->  {public_url}/")
print(f"  Patient Signup ->  {public_url}/patient/signup")
print(f"  Patient Login  ->  {public_url}/patient/login")
print(f"  Doctor Signup  ->  {public_url}/doctor/signup")
print(f"  Doctor Login   ->  {public_url}/doctor/login")
print(f"  Admin Panel    ->  {public_url}/admin/login")
print("=" * 65)
print("  Admin:  username=admin        password=admin123")
print("  Admin:  username=doctor_admin password=doctor2026")
print("=" * 65)
print("\n  All systems running! Keep this cell alive.")
print("  Closing or restarting this cell stops the server.")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5050
 * Running on http://172.28.0.12:5050
INFO:werkzeug:Press CTRL+C to quit



  Homepage       ->  https://f696-8-229-173-25.ngrok-free.app/
  Patient Signup ->  https://f696-8-229-173-25.ngrok-free.app/patient/signup
  Patient Login  ->  https://f696-8-229-173-25.ngrok-free.app/patient/login
  Doctor Signup  ->  https://f696-8-229-173-25.ngrok-free.app/doctor/signup
  Doctor Login   ->  https://f696-8-229-173-25.ngrok-free.app/doctor/login
  Admin Panel    ->  https://f696-8-229-173-25.ngrok-free.app/admin/login
  Admin:  username=admin        password=admin123
  Admin:  username=doctor_admin password=doctor2026

  All systems running! Keep this cell alive.
  Closing or restarting this cell stops the server.
